# Start

In [ ]:
%pwd

In [ ]:
!conda install -c conda-forge netcdf4 h5netcdf zarr -y

In [ ]:
conda list

In [ ]:
import os

os.environ["ESMF_MPILIB"] = "mpiuni"
os.environ["OMPI_MCA_btl"] = "^openib"
os.environ["UCX_TLS"] = "sm,self"

print("Set ESMF env vars")

In [ ]:
import os
print("ESMF_MPILIB =", os.environ.get("ESMF_MPILIB"))
print("OMPI_MCA_btl =", os.environ.get("OMPI_MCA_btl"))
print("UCX_TLS =", os.environ.get("UCX_TLS"))

import xesmf, esmpy
print("xesmf:", xesmf.__version__)
print("esmpy:", esmpy.__version__)

# # tiny regrid
# import xarray as xr, numpy as np
# lon = np.linspace(0, 10, 5)
# lat = np.linspace(0, 10, 4)
# ds1 = xr.Dataset(coords={"lon": lon, "lat": lat},
#                  data_vars={"a": (("lat","lon"), np.random.rand(4,5))})
# ds2 = xr.Dataset(coords={"lon": lon+0.1, "lat": lat+0.1})
# r = xesmf.Regridder(ds1, ds2, "bilinear", reuse_weights=False)
# out = r(ds1)
# print("Regridding OK:", out["a"].shape)

## Inspect

In [ ]:
import xarray as xr
import numpy as np

ds = xr.open_dataset('/mnt/data/khaiht/data/vietnam/output/era5_land_tp_2017_01.nc')
print(ds['tp'].attrs)  # look for "long_name" and "units"

# Pick any land point and print the first 60 hourly values
coord_name = 'valid_time' if 'valid_time' in ds.coords else 'time'
times = ds[coord_name].values
tp = ds['tp'].values

lat_idx, lon_idx = 100, 80
vals = tp[:, lat_idx, lon_idx]

print("\nRaw tp values for first 30 hours:")
for i in range(60):
    print(f"  {str(times[i])[:19]}  tp = {vals[i]:.6f} m")

In [ ]:
import xarray as xr
import numpy as np

ds = xr.open_dataset('/mnt/data/khaiht/data/vietnam/input_tp/era5_sl_tp_2017.nc')
print(ds['tp'].attrs)  # look for "long_name" and "units"

# Pick any land point and print the first 60 hourly values
coord_name = 'valid_time' if 'valid_time' in ds.coords else 'time'
times = ds[coord_name].values
tp = ds['tp'].values

lat_idx, lon_idx = 100, 80
vals = tp[:, lat_idx, lon_idx]

print("\nRaw tp values for first 30 hours:")
for i in range(60):
    print(f"  {str(times[i])[:19]}  tp = {vals[i]:.6f} m")

## Download script

### Target dataset

In [ ]:
import cdsapi
from pathlib import Path

client = cdsapi.Client()

dataset = "reanalysis-era5-land"

variables = {
    # "2m_temperature": "t2m",
    "total_precipitation": "tp",
}

area = [25, 100, 5, 125]  # N, W, S, E

out_dir = Path("/mnt/data/khaiht/data/vietnam/output")
out_dir.mkdir(parents=True, exist_ok=True)

# ── Main data: 2017-01 → 2025-12 ─────────────────────────────────────────
for year in range(2017, 2026):
    for month in range(1, 13):
        for var, short in variables.items():
            out_file = out_dir / f"era5_land_{short}_{year}_{month:02d}.nc"

            if out_file.exists():
                print(f"[SKIP] {out_file}")
                continue

            request = {
                "variable": var,
                "year": str(year),
                "month": f"{month:02d}",
                "day": [f"{d:02d}" for d in range(1, 32)],
                "time": [f"{h:02d}:00" for h in range(24)],
                "area": area,
                "data_format": "netcdf",
            }

            print(f"Downloading {out_file}")
            client.retrieve(dataset, request, str(out_file))

# ── Boundary months: Dec of the year BEFORE each split start ─────────────
# We need the last timestep of December (23:00) to correctly difference
# the very first hour (00:00) of January for each split:
#   • Training starts 2017-01-01 00:00  →  need 2016-12 (last ts: 2016-12-31 23:00)
#   • Test     starts 2025-01-01 00:00  →  need 2024-12 (already downloaded above)
#
# 2024-12 is already in the main loop above.
# Only 2016-12 needs a separate download.
boundary_months = [
    (2016, 12, "tp", "total_precipitation"),
]

for year, month, short, var in boundary_months:
    out_file = out_dir / f"era5_land_{short}_{year}_{month:02d}.nc"

    if out_file.exists():
        print(f"[SKIP] {out_file}")
    else:
        request = {
            "variable": var,
            "year": str(year),
            "month": f"{month:02d}",
            "day": [f"{d:02d}" for d in range(1, 32)],
            "time": [f"{h:02d}:00" for h in range(24)],
            "area": area,
            "data_format": "netcdf",
        }
        print(f"Downloading boundary month {out_file}")
        client.retrieve(dataset, request, str(out_file))


In [ ]:
from pathlib import Path
import zipfile
import shutil

ERA5_LAND_DIR = Path("/mnt/data/khaiht/data/vietnam/output")

def is_zip_file(path):
    with open(path, "rb") as f:
        return f.read(4) == b"PK\x03\x04"

fixed = 0
skipped = 0
failed = []

for nc_file in sorted(ERA5_LAND_DIR.glob("era5_land_*.nc")):
    try:
        if not is_zip_file(nc_file):
            print(f"[OK] {nc_file.name} already NetCDF")
            skipped += 1
            continue

        tmp_dir = nc_file.with_suffix("")  # e.g. era5_land_t2m_2019_01/
        tmp_dir.mkdir(exist_ok=True)

        with zipfile.ZipFile(nc_file, "r") as z:
            z.extractall(tmp_dir)

        # CDS almost always uses data_0.nc
        extracted = list(tmp_dir.glob("*.nc"))
        if len(extracted) != 1:
            raise RuntimeError(f"Expected 1 nc, found {len(extracted)}")

        real_nc = extracted[0]

        # Replace the broken file atomically
        shutil.move(real_nc, nc_file)

        # Cleanup
        shutil.rmtree(tmp_dir)

        print(f"[FIXED] {nc_file.name}")
        fixed += 1

    except Exception as e:
        print(f"[FAILED] {nc_file.name} → {e}")
        failed.append(nc_file.name)

print("\nSummary")
print("-------")
print(f"Fixed:   {fixed}")
print(f"Skipped: {skipped}")
print(f"Failed:  {len(failed)}")

### Input dataset

In [ ]:
import cdsapi
from pathlib import Path
import traceback

# =========================================================
# ERA5 Downloader
# Designed for SLURM batch execution
# =========================================================

client = cdsapi.Client()

# =========================
# Common settings
# =========================
years = list(range(2017, 2026))

# [North, West, South, East]
area = [30, 95, 0, 135]

months = [f"{m:02d}" for m in range(1, 13)]
days = [f"{d:02d}" for d in range(1, 32)]
times = [f"{h:02d}:00" for h in range(24)]

out_root = Path("/mnt/data/khaiht/data/vietnam/input_tp")
out_root.mkdir(parents=True, exist_ok=True)

# =========================================================
# Helper
# =========================================================
def should_download(path: Path, min_size_mb=50):
    """
    Skip if file exists and looks valid.
    min_size_mb protects against corrupted partial files.
    """
    if not path.exists():
        return True

    size_mb = path.stat().st_size / (1024**2)

    if size_mb < min_size_mb:
        print(f"⚠️ Small/corrupted file detected ({size_mb:.1f} MB)")
        print(f"🔄 Re-downloading: {path.name}")
        return True

    print(f"✅ Exists, skipping: {path.name} ({size_mb:.1f} MB)")
    return False


def download_single_level(var_long, var_short, year):
    out_file = out_root / f"era5_sl_{var_short}_{year}.nc"

    if not should_download(out_file):
        return

    request = {
        "product_type": "reanalysis",
        "variable": var_long,
        "year": str(year),
        "month": months,
        "day": days,
        "time": times,
        "area": area,
        "data_format": "netcdf",
    }

    print("=" * 60)
    print(f"⬇️ Downloading: {out_file.name}")
    print("=" * 60)

    client.retrieve(
        "reanalysis-era5-single-levels",
        request,
        str(out_file),
    )

    print(f"✅ Finished: {out_file.name}")


def download_pressure_level(var_long, short_name, level, year):
    out_file = out_root / f"era5_pl_{short_name}_{year}.nc"

    if not should_download(out_file):
        return

    request = {
        "product_type": "reanalysis",
        "variable": var_long,
        "pressure_level": level,
        "year": str(year),
        "month": months,
        "day": days,
        "time": times,
        "area": area,
        "data_format": "netcdf",
    }

    print("=" * 60)
    print(f"⬇️ Downloading: {out_file.name}")
    print("=" * 60)

    client.retrieve(
        "reanalysis-era5-pressure-levels",
        request,
        str(out_file),
    )

    print(f"✅ Finished: {out_file.name}")


# =========================================================
# Variables
# =========================================================

single_level_vars = {
    # "mean_sea_level_pressure": "msl",
    # "10m_u_component_of_wind": "u10",
    # "10m_v_component_of_wind": "v10",
    # "total_column_water_vapour": "tcwv",
    # "2m_temperature": "t2m",
    # "2m_dewpoint_temperature": "d2m",
    "total_precipitation": "tp",
}

pressure_level_vars = {
    "geopotential": {"short": "z500", "level": "500"},
    "temperature": {"short": "t850", "level": "850"},
    "specific_humidity": {"short": "q850", "level": "850"},
    "u_component_of_wind": {"short": "u850", "level": "850"},
    "v_component_of_wind": {"short": "v850", "level": "850"},
    "vertical_velocity": {"short": "omega500", "level": "500"},
}


# =========================================================
# Main
# =========================================================
def main():

    # -------------------------
    # Single-level variables
    # -------------------------
    for var_long, var_short in single_level_vars.items():
        for year in years:
            try:
                download_single_level(var_long, var_short, year)

            except Exception as e:
                print(f"❌ Failed: {var_long} {year}")
                print(str(e))
                traceback.print_exc()

    # -------------------------
    # Pressure-level variables
    # Uncomment if needed
    # -------------------------
    """
    for var_long, info in pressure_level_vars.items():
        for year in years:
            try:
                download_pressure_level(
                    var_long,
                    info["short"],
                    info["level"],
                    year
                )

            except Exception as e:
                print(f"❌ Failed: {var_long} {year}")
                print(str(e))
                traceback.print_exc()
    """

    print("\n🎉 All downloads complete.")


if __name__ == "__main__":
    main()

## General

In [ ]:
LAT_MIN, LAT_MAX = 5.8, 25
LON_MIN, LON_MAX = 102, 118

In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path

# ------------------------
# CONFIG
# ------------------------
RAW_INPUT  = Path("/mnt/data/khaiht/data/vietnam/input")
RAW_OUTPUT = Path("/mnt/data/khaiht/data/vietnam/output")

PROCESSED_INPUT  = Path("/mnt/data/khaiht/data/vietnamvip_processed/input")
PROCESSED_OUTPUT = Path("/mnt/data/khaiht/data/vietnamvip_processed/output")

PROCESSED_INPUT.mkdir(parents=True, exist_ok=True)
PROCESSED_OUTPUT.mkdir(parents=True, exist_ok=True)

# ------------------------
# HELPERS
# ------------------------
def crop(ds):
    return ds.sel(
        latitude=slice(LAT_MAX, LAT_MIN),
        longitude=slice(LON_MIN, LON_MAX),
    )

def clean_ds(ds):
    print("  [DEBUG] Original dims:", ds.dims)

    if "valid_time" in ds.dims or "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})

    for dim in ["expver", "number"]:
        if dim in ds.dims:
            ds = ds.isel({dim: 0}, drop=True)
        if dim in ds.coords:
            ds = ds.drop_vars(dim)

    ds.attrs = {}
    for v in ds.variables:
        ds[v].attrs = {}

    print("  [DEBUG] Cleaned dims:", ds.dims)
    return ds

def era5_land_tp_to_hourly(tp_da):
    """
    Convert ERA5-Land *accumulated* TP to true hourly TP (mm), then log1p.

    ERA5-Land tp resets once per day at 01:00 UTC.
    Accumulation pattern within each run:
        01:00  reset — value = precip in that 1 hour only
        02:00  accumulated since 01:00
        ...
        00:00 next day — accumulated since yesterday 01:00  (still rising)

    Input contract:
        tp_da must have N timesteps where tp_da[0] is the boundary (T-1)
        value prepended by the caller.  The function returns N-1 timesteps
        (the boundary hour is consumed by the diff and never appears in output).

    Differencing rule:
        hourly[t] = tp[t] - tp[t-1]    when tp[t] - tp[t-1] >= -RESET_THRESHOLD_M  (normal)
        hourly[t] = tp[t]               when tp[t] - tp[t-1] <  -RESET_THRESHOLD_M  (reset hour)

    Float32 precision note
    ----------------------
    ERA5-Land TP is stored as float32.  At a plateau value of ~0.144 m the
    machine epsilon is ~1.7e-8 m (~0.017 µm).  When the accumulator stops
    changing (no rain), consecutive timesteps can differ by ±1–2 ULPs and
    the diff can be a tiny *negative* number even though no reset occurred.

    Using `tp_diff < 0` as the reset condition incorrectly treats this noise
    as a reset and substitutes the full raw_accum value (~144 mm) as the
    "hourly" rate — producing the spurious 144 mm/hr spike seen in the plot.

    The fix is a negative threshold of -1e-5 m (-0.01 mm):
        • float32 noise at 0.144 m  ≈  ±1.7e-8 m  →  safely ABOVE threshold
        • smallest real reset         ≈    −0.1 mm  →  safely BELOW threshold
        • typical large reset         ≈   −100 mm   →  far BELOW threshold

    Any sub-threshold negative diffs that slip through are zeroed by .clip(min=0).

    Returns a DataArray whose time coordinate starts at tp_da.time[1]
    (i.e. the first real data hour, NOT the boundary hour).
    """
    # diff along time → length N-1, time coord = tp_da.time[1:]
    tp_diff = tp_da.diff(dim="time")

    # ── CRITICAL FIX ────────────────────────────────────────────────────────
    # Use a physically meaningful negative threshold instead of bare `< 0`.
    # A true daily reset drops by tens-to-hundreds of mm; float32 noise at
    # a ~144 mm plateau is only ~1e-8 m.  The old `< 0` condition
    # misidentified float32 noise as a reset and injected ~144 mm/hr spikes.
    RESET_THRESHOLD_M = 1e-5   # 0.01 mm in metres — 3 orders of magnitude
                                # above float32 noise, far below any real reset
    # ────────────────────────────────────────────────────────────────────────

    tp_hourly = xr.where(
        tp_diff < -RESET_THRESHOLD_M,
        tp_da.isel(time=slice(1, None)),   # raw value at reset hour
        tp_diff,
    )
    # tp_hourly.time == tp_da.time[1:] — boundary hour is gone ✓

    # m → mm, clip any sub-threshold float32 negatives to zero, then log1p
    tp_hourly = (tp_hourly * 1000.0).clip(min=0.0)

    print("  [DEBUG] Hourly TP (mm) before log1p: min/max =",
          float(tp_hourly.min()), float(tp_hourly.max()))

    tp_hourly = np.log1p(tp_hourly)

    print("  [DEBUG] Hourly TP after log1p:        min/max =",
          float(tp_hourly.min()), float(tp_hourly.max()))

    return tp_hourly

## Target dataset

In [ ]:
# ------------------------
# ERA5-LAND  (accumulated → true hourly TP, month-by-month)
# ------------------------
import pandas as pd

for year in range(2017, 2026):
    monthly_hourly = []
    monthly_stats = []

    for month in range(1, 13):
        if month == 1:
            prev_file = RAW_OUTPUT / f"era5_land_tp_{year-1}_12.nc"
        else:
            prev_file = RAW_OUTPUT / f"era5_land_tp_{year}_{month-1:02d}.nc"

        ds_prev = xr.open_dataset(prev_file)
        ds_prev = clean_ds(ds_prev)
        ds_boundary = ds_prev.isel(time=[-1])

        ds_month = xr.open_dataset(
            RAW_OUTPUT / f"era5_land_tp_{year}_{month:02d}.nc"
        )
        ds_month = clean_ds(ds_month)

        ds_extended = xr.concat([ds_boundary, ds_month], dim="time")
        ds_extended = crop(ds_extended)

        tp_hourly = era5_land_tp_to_hourly(ds_extended["tp"])
        monthly_hourly.append(tp_hourly)

        # Stats: undo log1p to get real mm/hr
        tp_mm = np.expm1(tp_hourly)
        max_val = float(tp_mm.max())

        # Find the single argmax index across all dimensions
        flat_idx = tp_mm.argmax(dim=["time", "latitude", "longitude"])
        max_time = tp_mm.time.values[int(flat_idx["time"])]
        max_lat  = float(tp_mm.latitude.values[int(flat_idx["latitude"])])
        max_lon  = float(tp_mm.longitude.values[int(flat_idx["longitude"])])

        monthly_stats.append({
            "Month":      pd.Timestamp(f"{year}-{month:02d}-01").strftime("%b"),
            "Max mm/hr":  round(max_val, 2),
            "At (UTC)":   str(max_time)[:16].replace("T", " "),
            "Lat":        round(max_lat, 2),
            "Lon":        round(max_lon, 2),
        })

        ds_prev.close()
        ds_month.close()

    # Concatenate all 12 months
    tp_year = xr.concat(monthly_hourly, dim="time")

    assert str(tp_year.time.values[0])[:10] == f"{year}-01-01", \
        f"Unexpected first timestep: {tp_year.time.values[0]}"
    assert tp_year.sizes["time"] == (8784 if year in (2020, 2024) else 8760), \
        f"Unexpected time length: {tp_year.sizes['time']}"

    ds_out = xr.Dataset({"tp": tp_year}).astype("float32")
    out_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_out.to_netcdf(out_file)

    # ── Clean summary table ───────────────────────────────────────────────
    year_max = float(np.expm1(tp_year).max())
    print(f"\n{'='*65}")
    print(f"  ERA5-Land Hourly TP — {year}  [{tp_year.sizes['time']} timesteps]")
    print(f"{'='*65}")
    print(f"  {'Month':<6} {'Max (mm/hr)':>12}  {'At (UTC)':<17} {'Lat':>7} {'Lon':>8}")
    print(f"  {'-'*6} {'-'*12}  {'-'*16} {'-'*7} {'-'*8}")
    for r in monthly_stats:
        print(f"  {r['Month']:<6} {r['Max mm/hr']:>12.2f}  {r['At (UTC)']:<17} {r['Lat']:>7.2f} {r['Lon']:>8.2f}")
    print(f"  {'-'*6} {'-'*12}")
    print(f"  {'ANNUAL':6} {year_max:>12.2f}")
    print(f"\n  ✅ Saved {out_file}")

### Check

In [ ]:
"""
ERA5-Land TP Preprocessing — Robust Verification Suite
=======================================================
Replaces the old suite that crashed at Check 2 due to:
  • Checks 1 & 2 each reopening and re-reading all raw files independently
    (2× peak memory, 2× I/O, kernel OOM on large domains)
  • A numpy nanargmin bug in the "worst noise examples" collector
  • No file-by-file garbage collection (xarray datasets left open)

Key design decisions
---------------------
  • Checks 1 & 2 share a SINGLE pass over raw files (one open per month).
  • All heavy arrays (tp diffs) are computed in numpy and immediately freed;
    no xarray lazy graphs are held in memory across months.
  • "Worst noise examples" collected with a bounded heap (no full-sort).
  • Check 4 (mass-balance) opens raw files only for the specific timestamps
    it needs, not the whole month at once.
  • Each check returns (passed: bool, detail: dict) so callers can gate
    downstream checks or log results programmatically.

Checks
------
  1+2  False-reset audit & reset-hour audit  (single raw-file pass)
  3    Spike scan on processed files
  4    Mass-balance: sum(hourly window) ≈ raw_accum at window end
  5    Timestamp alignment: raw and processed time axes agree
  6    Distribution sanity table (annual max / p99 / p99.9 / wet-fraction)
     + Visual verification: 3-panel plot for a chosen year/month
"""

import heapq
import gc
from collections import Counter
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_OUTPUT       = Path("/mnt/data/khaiht/data/vietnam/output")
PROCESSED_OUTPUT = Path("/mnt/data/khaiht/data/vietnamvip_processed/output")

LAT_MIN, LAT_MAX = 5.8, 25.0
LON_MIN, LON_MAX = 102.0, 118.0

YEARS  = list(range(2017, 2026))
MONTHS = list(range(1, 13))

# Must match era5_land_tp_to_hourly exactly
RESET_THRESHOLD_M  = 1e-5   # metres  (0.01 mm)

# Physical plausibility ceiling for hourly rainfall
SPIKE_THRESHOLD_MM = 350.0  # mm/hr — world record ~305 mm/hr


# ══════════════════════════════════════════════════════════════════════════
# Shared helpers
# ══════════════════════════════════════════════════════════════════════════

def _clean_ds(ds: xr.Dataset) -> xr.Dataset:
    if "valid_time" in ds.dims or "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    for dim in ("expver", "number"):
        if dim in ds.dims:
            ds = ds.isel({dim: 0}, drop=True)
        if dim in ds.coords:
            ds = ds.drop_vars(dim)
    ds.attrs = {}
    for v in ds.variables:
        ds[v].attrs = {}
    return ds


def _crop(ds: xr.Dataset) -> xr.Dataset:
    return ds.sel(
        latitude=slice(LAT_MAX, LAT_MIN),
        longitude=slice(LON_MIN, LON_MAX),
    )


def _sep(title: str = "", width: int = 68) -> None:
    bar = "═" * width
    print(f"\n{bar}")
    if title:
        print(f"  {title}")
        print(bar)


def _load_raw_np(year: int, month: int):
    """
    Open one raw monthly file, crop, return (times_np, tp_np, ds) where
    tp_np is a float32 C-contiguous numpy array shaped (T, lat, lon).
    Caller is responsible for ds.close().
    Returns None, None, None if file missing.
    """
    raw_file = RAW_OUTPUT / f"era5_land_tp_{year}_{month:02d}.nc"
    if not raw_file.exists():
        print(f"  [SKIP] {raw_file.name} not found")
        return None, None, None
    ds = xr.open_dataset(raw_file)
    ds = _clean_ds(ds)
    ds = _crop(ds)
    tp_np    = ds["tp"].values.astype(np.float32)   # load fully, then free xarray lazy ref
    times_np = ds["time"].values
    return times_np, tp_np, ds


# ══════════════════════════════════════════════════════════════════════════
# CHECK 1 + 2 — Single pass over raw files
# ══════════════════════════════════════════════════════════════════════════

def check1_and_2_combined(
    years=YEARS,
    months=MONTHS,
    top_n: int = 20,
) -> tuple[bool, bool]:
    """
    Single pass: for every raw monthly file compute tp_diff once and
    extract both the false-reset (Check 1) and reset-hour (Check 2) stats.

    Returns
    -------
    (check1_passed, check2_passed) — both True means no problems found.
    """
    _sep("CHECK 1 — False-reset audit  +  CHECK 2 — Reset-hour audit  (single pass)")

    total_cell_steps   = 0
    genuine_resets     = 0
    noise_false_resets = 0

    # Bounded heap for top-N worst noise diffs: stores (-|diff|, year, mon, t_str, lat, lon)
    worst_heap: list = []     # min-heap of size top_n on -|diff|

    noise_hour_counter: Counter = Counter()

    # Check 2 accumulators
    total_reset_tsteps = 0
    non_01_resets: list[tuple] = []  # (year, month, time_str, hour)

    for year in years:
        for month in months:
            times_np, tp_np, ds = _load_raw_np(year, month)
            if ds is None:
                continue

            # Shape: (T-1, lat, lon)
            diff = np.diff(tp_np, axis=0)          # float32 in-place diff
            T_minus1, nlat, nlon = diff.shape
            total_cell_steps += T_minus1 * nlat * nlon

            # ── Check 1 masks ──────────────────────────────────────────
            genuine_mask = diff < -RESET_THRESHOLD_M          # (T-1, lat, lon) bool
            noise_mask   = (diff < 0) & ~genuine_mask         # idem

            genuine_resets    += int(genuine_mask.sum())
            n_noise            = int(noise_mask.sum())
            noise_false_resets += n_noise

            # Collect noise hour distribution + worst examples
            if n_noise > 0:
                # time indices where any pixel has noise (spatial collapse)
                noise_any_spatial = noise_mask.any(axis=(1, 2))   # (T-1,)
                noise_time_idxs   = np.where(noise_any_spatial)[0]

                # diff times = times_np[1:]  (diff[i] = tp[i+1] - tp[i])
                diff_times = times_np[1:]
                for ti in noise_time_idxs:
                    h = pd.Timestamp(diff_times[ti]).hour
                    noise_hour_counter[h] += 1

                # ── Collect top-N worst noise examples (enriched tuples only) ──
                # Strategy: find the top_n most-negative noise values via
                # np.partition (O(n)), then walk only those candidates to
                # attach spatial metadata.  The heap always contains full
                # (v_val, year, month, t_str, lat, lon) tuples — no bare floats.
                flat_2d = diff.reshape(T_minus1, -1)          # (T-1, lat*lon)

                # Flatten only the noise pixels to find the severity threshold
                noise_flat = flat_2d[noise_mask.reshape(T_minus1, -1)]
                k = min(top_n, len(noise_flat))
                # np.partition: the k smallest (most-negative) values end up in [:k]
                threshold_v = float(np.partition(noise_flat, k - 1)[k - 1])

                for ti in range(T_minus1):
                    row = flat_2d[ti]
                    # Only inspect pixels at or below the severity threshold
                    cands = np.where(row <= threshold_v)[0]
                    for ci in cands:
                        v_val = float(diff[ti, ci // nlon, ci % nlon])
                        # Must be noise: negative but above -RESET_THRESHOLD_M
                        if v_val >= 0 or v_val < -RESET_THRESHOLD_M:
                            continue
                        lai   = ci // nlon
                        loi   = ci % nlon
                        lat   = float(ds["latitude"].values[lai])
                        lon   = float(ds["longitude"].values[loi])
                        t_str = str(diff_times[ti])[:16]
                        entry = (v_val, year, month, t_str, lat, lon)
                        if len(worst_heap) < top_n:
                            heapq.heappush(worst_heap, entry)
                        elif v_val < worst_heap[0][0]:   # more negative = larger |diff|
                            heapq.heapreplace(worst_heap, entry)

            # ── Check 2 ────────────────────────────────────────────────
            genuine_any = genuine_mask.any(axis=(1, 2))   # (T-1,) bool
            reset_time_idxs = np.where(genuine_any)[0]
            diff_times = times_np[1:]

            total_reset_tsteps += len(reset_time_idxs)
            for ti in reset_time_idxs:
                h = pd.Timestamp(diff_times[ti]).hour
                if h != 1:
                    non_01_resets.append((year, month, str(diff_times[ti])[:16], h))

            # ── Free memory ────────────────────────────────────────────
            ds.close()
            del diff, tp_np, times_np, genuine_mask, noise_mask
            gc.collect()

    # ── CHECK 1 report ─────────────────────────────────────────────────────
    _sep("CHECK 1 results")
    print(f"  Total cell-steps scanned : {total_cell_steps:,}")
    print(f"  Genuine resets detected  : {genuine_resets:,}")
    print(f"  Float32 noise false-resets (OLD bug): {noise_false_resets:,}")

    check1_passed = noise_false_resets == 0

    if check1_passed:
        print("  ✅  No false resets found — processed files should be clean.")
    else:
        print(f"\n  ⚠️  {noise_false_resets:,} false resets would have been injected by old code!")

        # Reconstruct worst list from heap (always full tuples)
        worst_list = sorted(worst_heap, key=lambda x: x[0])  # most negative first
        if worst_list:
            print(f"\n  Top-{min(top_n, len(worst_list))} worst noise diffs:")
            print(f"  {'|diff| (m)':<14} {'year':>4} {'mon':>3}  {'time':<17}  {'lat':>7} {'lon':>7}")
            print(f"  {'-'*14} {'-'*4} {'-'*3}  {'-'*16}  {'-'*7} {'-'*7}")
            for v, yr, mo, t_str, lat, lon in worst_list[:top_n]:
                print(
                    f"  {abs(v):.4e}     {yr:>4} {mo:>3}  {t_str:<17}  "
                    f"{lat:>7.2f} {lon:>7.2f}"
                )

        if noise_hour_counter:
            print(f"\n  UTC-hour distribution of false resets:")
            max_count = max(noise_hour_counter.values())
            for h in range(24):
                cnt = noise_hour_counter.get(h, 0)
                bar = "█" * int(cnt / max_count * 40)
                print(f"  {h:02d}:00  {cnt:>6,}  {bar}")

    # ── CHECK 2 report ─────────────────────────────────────────────────────
    _sep("CHECK 2 results — Reset-hour audit (all genuine resets must be 01:00 UTC)")
    print(f"  Total reset timesteps found : {total_reset_tsteps:,}")
    check2_passed = len(non_01_resets) == 0
    if check2_passed:
        print("  ✅  All genuine resets are at 01:00 UTC.")
    else:
        print(f"  ⚠️  {len(non_01_resets)} reset(s) at unexpected hours:")
        print(f"  {'year':>4} {'mon':>3}  {'time':<17}  {'hour':>4}")
        for yr, mo, t_str, h in non_01_resets[:30]:
            print(f"  {yr:>4} {mo:>3}  {t_str:<17}  {h:>4}")
        if len(non_01_resets) > 30:
            print(f"  … ({len(non_01_resets) - 30} more not shown)")

    return check1_passed, check2_passed


# ══════════════════════════════════════════════════════════════════════════
# CHECK 3 — Spike scan on processed files
# ══════════════════════════════════════════════════════════════════════════

def check3_spike_scan(years=YEARS, threshold_mm=SPIKE_THRESHOLD_MM) -> bool:
    _sep(f"CHECK 3 — Spike scan on processed files (threshold = {threshold_mm} mm/hr)")

    all_spikes = []

    for year in years:
        proc_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
        if not proc_file.exists():
            print(f"  [SKIP] {proc_file.name} not found")
            continue

        ds = xr.open_dataset(proc_file)
        tp_np = np.expm1(ds["tp"].values.astype(np.float64))  # mm/hr
        times = ds["time"].values
        lats  = ds["latitude"].values
        lons  = ds["longitude"].values
        ds.close()

        spike_idxs = np.argwhere(tp_np > threshold_mm)
        n_spikes = len(spike_idxs)

        if n_spikes > 0:
            print(f"  ⚠️  {year}: {n_spikes} spike(s) above {threshold_mm} mm/hr")
            for ti, lai, loi in spike_idxs[:10]:
                val = tp_np[ti, lai, loi]
                t_str = str(times[ti])[:16]
                lat   = float(lats[lai])
                lon   = float(lons[loi])
                all_spikes.append((year, t_str, lat, lon, val))
                print(f"     {t_str}  lat={lat:.2f}  lon={lon:.2f}  {val:.2f} mm/hr")
        else:
            print(f"  ✅  {year}: no spikes above {threshold_mm} mm/hr")

        del tp_np
        gc.collect()

    passed = len(all_spikes) == 0
    if passed:
        print("\n  ✅  All processed files pass the spike scan.")
    else:
        print(f"\n  ⚠️  Total spikes found: {len(all_spikes)}")
    return passed


# ══════════════════════════════════════════════════════════════════════════
# CHECK 4 — Mass-balance check
# ══════════════════════════════════════════════════════════════════════════

def check4_mass_balance(years=YEARS, n_points: int = 5, tol_mm: float = 0.5) -> bool:
    """
    For n_points random grid cells per year, verify mass conservation:

        sum(hourly_mm[01:00 .. 00:00_next])
            ≈ raw_accum[00:00_next] − raw_accum[00:00_this_day]

    Why the difference, not the absolute?
    ─────────────────────────────────────
    ERA5-Land resets at 01:00 UTC each day, so raw_accum[00:00] is the
    *total accumulation since the previous 01:00*, which equals the raw
    value at 00:00 (end of the window) MINUS the raw value at 00:00 of
    the same calendar day (which is the carry-over from the prior run,
    i.e. the last point before the reset).

    In practice raw_accum[00:00_this_day] is whatever was left at the end
    of the previous day's run — often 0 after a dry stretch, but non-zero
    after overnight rain.  Using only raw_accum[00:00_next] as the
    reference double-counts that carry-over and produces the ~2 mm errors
    seen at wet grid cells.

    Window skipping rules
    ─────────────────────
    • Skip if end index is out of range or doesn't land at 00:00.
    • Skip the very first window of the year: the 01:00 start is Jan-1
      01:00 but the 00:00 reference for the same day is Dec-31 00:00
      of the *previous* year's raw file — it's valid but we'd need to
      open an extra file; simpler to skip.
    • Skip cross-month windows where start and end come from different
      raw files (handled naturally: both timestamps looked up independently).
    """
    _sep(f"CHECK 4 — Mass-balance (sum of hourly ≈ Δraw daily accum, tol={tol_mm} mm)")

    rng      = np.random.default_rng(42)
    all_pass = True

    # Cache for raw point-series within a single year run (keyed by (year, month))
    _raw_cache: dict = {}

    def _get_raw_point(yr, mo, lat, lon):
        """Return (times_np, accum_mm_1d) for one grid point, cached per month."""
        key = (yr, mo, round(lat, 4), round(lon, 4))
        if key in _raw_cache:
            return _raw_cache[key]
        raw_file = RAW_OUTPUT / f"era5_land_tp_{yr}_{mo:02d}.nc"
        if not raw_file.exists():
            _raw_cache[key] = (None, None)
            return None, None
        ds = xr.open_dataset(raw_file)
        ds = _clean_ds(ds)
        times = ds["time"].values
        vals  = (
            ds["tp"]
            .sel(latitude=lat, longitude=lon, method="nearest")
            .values.astype(np.float64)
        ) * 1000.0  # m → mm
        ds.close()
        _raw_cache[key] = (times, vals)
        return times, vals

    for year in years:
        proc_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
        if not proc_file.exists():
            continue

        _raw_cache.clear()   # reset cache per year to bound memory

        ds_proc    = xr.open_dataset(proc_file)
        tp_proc_mm = np.expm1(ds_proc["tp"].values.astype(np.float64))
        proc_times = ds_proc["time"].values
        proc_lats  = ds_proc["latitude"].values
        proc_lons  = ds_proc["longitude"].values
        ds_proc.close()

        lat_idxs = rng.integers(0, len(proc_lats), n_points)
        lon_idxs = rng.integers(0, len(proc_lons), n_points)
        year_pass = True

        proc_hours = np.array([pd.Timestamp(t).hour for t in proc_times])
        reset_idxs = np.where(proc_hours == 1)[0]

        for pi in range(n_points):
            lat = float(proc_lats[lat_idxs[pi]])
            lon = float(proc_lons[lon_idxs[pi]])
            hourly = tp_proc_mm[:, lat_idxs[pi], lon_idxs[pi]]

            max_err   = 0.0
            n_windows = 0

            for idx, ri in enumerate(reset_idxs):
                # Skip the very first window of the year (would need prev-year raw)
                if idx == 0:
                    continue

                end = ri + 23
                if end >= len(proc_times):
                    break
                if pd.Timestamp(proc_times[end]).hour != 0:
                    continue  # incomplete window

                window_sum = float(hourly[ri:end + 1].sum())

                # Reference: raw_accum[00:00_end] − raw_accum[00:00_start_of_same_day]
                # 00:00 of the same calendar day as ri (= proc_times[ri] shifted back 1h)
                start_ts = pd.Timestamp(proc_times[ri])   # 01:00
                end_ts   = pd.Timestamp(proc_times[end])  # 00:00 next day

                # 00:00 of start_ts's calendar day = 1 hour before 01:00
                day_start_ts = start_ts - pd.Timedelta(hours=1)  # 00:00 same day

                # Fetch raw values (cached per month)
                t_end, v_end = _get_raw_point(end_ts.year,   end_ts.month,   lat, lon)
                t_ds,  v_ds  = _get_raw_point(day_start_ts.year, day_start_ts.month, lat, lon)

                if t_end is None or t_ds is None:
                    continue

                # Find nearest index in each raw time array
                def _nearest_val(times_arr, vals_arr, target_ts):
                    target_np = np.datetime64(target_ts)
                    idx_n = int(np.argmin(np.abs(times_arr - target_np)))
                    return vals_arr[idx_n]

                raw_end      = _nearest_val(t_end, v_end, end_ts)
                raw_day_zero = _nearest_val(t_ds,  v_ds,  day_start_ts)

                # Expected hourly sum = accumulation added during the window
                expected = raw_end - raw_day_zero

                # Guard: if raw resets within this window (negative expected),
                # the window spans a month boundary with unusual accumulation;
                # skip rather than flag falsely.
                if expected < -tol_mm:
                    continue

                err     = abs(window_sum - expected)
                max_err = max(max_err, err)
                n_windows += 1

            ok = max_err <= tol_mm
            status = "✅" if ok else "⚠️ "
            if not ok:
                year_pass = False
                all_pass  = False
            print(
                f"  {status} {year}  lat={lat:.2f} lon={lon:.2f}  "
                f"{n_windows} windows  max_err={max_err:.4f} mm"
            )

        if year_pass:
            print(f"     → {year} mass-balance OK ✅")

        del tp_proc_mm
        _raw_cache.clear()
        gc.collect()

    if all_pass:
        print("\n  ✅  All mass-balance checks passed.")
    return all_pass


# ══════════════════════════════════════════════════════════════════════════
# CHECK 5 — Timestamp alignment
# ══════════════════════════════════════════════════════════════════════════

def check5_timestamp_alignment(years=YEARS) -> bool:
    """
    Verify processed annual files have correct time axes.

    Expected first timestamp: {year}-01-01 00:00 UTC
    ────────────────────────────────────────────────────
    era5_land_tp_to_hourly prepends the last timestep of the previous
    month (Dec 31 23:00) as a boundary, then diffs.  The diff output
    starts at tp_da.time[1], which is Jan 1 00:00 of the raw file.
    So the first processed hour is always 00:00, NOT 01:00.

    Expected length: hours in the calendar year
    ────────────────────────────────────────────
    Leap years have 8784 h, common years 8760 h.  We derive this from
    pandas rather than hard-coding, to be safe.

    Additional checks
    ─────────────────
    • Last timestamp must be {year}-12-31 23:00 UTC.
    • No duplicate timestamps.
    • No gaps > 1 h (monotonically spaced at 1-hour intervals).
    """
    _sep("CHECK 5 — Timestamp alignment")

    all_pass = True

    for year in years:
        proc_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
        if not proc_file.exists():
            print(f"  [SKIP] {proc_file.name} not found")
            continue

        ds    = xr.open_dataset(proc_file)
        times = ds["time"].values
        ds.close()

        first_t = pd.Timestamp(times[0])
        last_t  = pd.Timestamp(times[-1])

        # Expected bounds
        exp_first = pd.Timestamp(f"{year}-01-01 00:00")
        exp_last  = pd.Timestamp(f"{year}-12-31 23:00")

        # Expected length from pandas date_range (handles leap years correctly)
        exp_len = len(pd.date_range(exp_first, exp_last, freq="h"))

        issues = []
        if first_t != exp_first:
            issues.append(f"first ts = {first_t} (expected {exp_first})")
        if last_t != exp_last:
            issues.append(f"last ts = {last_t} (expected {exp_last})")
        if len(times) != exp_len:
            issues.append(f"len={len(times)} (expected {exp_len})")

        # Check for duplicates and gaps (convert to int64 ns for fast diff)
        if len(times) > 1:
            dt_ns  = np.diff(times.astype("datetime64[ns]").astype(np.int64))
            one_h  = np.timedelta64(1, "h") / np.timedelta64(1, "ns")
            n_dups = int((dt_ns == 0).sum())
            n_gaps = int((dt_ns > one_h).sum())
            if n_dups:
                issues.append(f"{n_dups} duplicate timestamp(s)")
            if n_gaps:
                issues.append(f"{n_gaps} gap(s) > 1 h")

        if issues:
            all_pass = False
            for issue in issues:
                print(f"  ⚠️  {year}: {issue}")
        else:
            print(f"  ✅  {year}: {len(times)} steps, {first_t} → {last_t}")

    if all_pass:
        print("\n  ✅  All timestamp checks passed.")
    return all_pass


# ══════════════════════════════════════════════════════════════════════════
# CHECK 6 — Distribution sanity table
# ══════════════════════════════════════════════════════════════════════════

def check6_distribution(years=YEARS) -> bool:
    """
    Print annual statistics (max, p99, p99.9, wet-fraction) and flag
    any year whose max exceeds SPIKE_THRESHOLD_MM or whose wet-fraction
    is implausibly high (>0.9) / low (<0.001).

    NaN / fill-value handling
    ─────────────────────────
    ERA5-Land files may contain ocean/out-of-domain pixels stored as the
    netCDF _FillValue (typically 9.969e+36 or NaN after xarray decode).
    After expm1() a large-positive fill value becomes an even larger
    positive, poisoning mean/max/percentile.  We therefore:
      1. Cast to float32 first (matches stored precision), then float64.
      2. Mask out any value that expm1 maps to > SPIKE_THRESHOLD_MM *before*
         computing stats — those are either genuine spikes (caught by Check 3)
         or fill values.
      3. Use np.nan* variants throughout.
      4. Report the NaN/fill fraction so you know how much of the domain
         is ocean.
    """
    _sep("CHECK 6 — Distribution sanity table")

    header = (
        f"  {'Year':>4}  {'Hours':>6}  {'NaNfrac':>8}  {'Mean':>8}  "
        f"{'Max':>8}  {'p99':>8}  {'p99.9':>8}  {'WetFrac':>8}"
    )
    print(header)
    print("  " + "-" * (len(header) - 2))

    all_pass = True

    for year in years:
        proc_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
        if not proc_file.exists():
            continue

        ds = xr.open_dataset(proc_file)
        # Load as float32 (native), then upcast — avoids expm1 overflow on fill values
        raw_log = ds["tp"].values.astype(np.float32)
        ds.close()

        # Mask fill values in log1p space: valid log1p(mm) is in [0, ~log1p(350)] ≈ [0, 5.86]
        # Anything outside [-0.1, 10] in log space is a fill/corrupt value
        valid_mask = (raw_log >= -0.1) & (raw_log <= 10.0) & np.isfinite(raw_log)
        nan_frac   = float((~valid_mask).mean())

        tp_mm = np.where(valid_mask, np.expm1(raw_log.astype(np.float64)), np.nan)
        flat  = tp_mm.ravel()

        # Use nan-safe reductions
        mean_v   = float(np.nanmean(flat))
        max_v    = float(np.nanmax(flat))
        p99      = float(np.nanpercentile(flat, 99))
        p999     = float(np.nanpercentile(flat, 99.9))
        wet_frac = float(np.nanmean(flat > 0.1))
        n_steps  = tp_mm.shape[0]

        flags = []
        if max_v > SPIKE_THRESHOLD_MM:
            flags.append(f"max={max_v:.0f} > {SPIKE_THRESHOLD_MM}")
            all_pass = False
        if wet_frac > 0.9:
            flags.append(f"wet_frac={wet_frac:.3f} > 0.9")
            all_pass = False
        if wet_frac < 0.001:
            flags.append(f"wet_frac={wet_frac:.4f} < 0.001")
            all_pass = False
        if np.isnan(mean_v):
            flags.append("mean is NaN — all values masked?")
            all_pass = False

        flag_str = "  ⚠️  " + ", ".join(flags) if flags else ""
        print(
            f"  {year:>4}  {n_steps:>6}  {nan_frac:>8.4f}  {mean_v:>8.4f}  "
            f"{max_v:>8.2f}  {p99:>8.2f}  {p999:>8.2f}  {wet_frac:>8.4f}{flag_str}"
        )

        del tp_mm, flat, raw_log
        gc.collect()

    if all_pass:
        print("\n  ✅  All years pass distribution sanity checks.")
    return all_pass


# ══════════════════════════════════════════════════════════════════════════
# VISUAL VERIFICATION — 3-panel spot-check plot
# ══════════════════════════════════════════════════════════════════════════

def visual_verification(
    check_year: int  = 2020,
    check_month: int = 8,
    n_days: int      = 5,
) -> None:
    """
    3-panel verification plot centred on the peak hourly rainfall event
    within check_year / check_month:
 
      Panel 1 — raw accumulated TP (mm) with daily reset markers
      Panel 2 — processed hourly TP (mm/hr)
      Panel 3 — piecewise cumulative processed vs raw delta (mass-balance)
 
    The window is  [peak − floor(n_days/2) days,  peak + ceil(n_days/2) days),
    snapped to whole-day boundaries so reset markers always align cleanly.
    If the window extends into the adjacent month the raw data is loaded from
    both monthly files and concatenated.
    """
    half_before = n_days // 2
    half_after  = n_days - half_before
 
    proc_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{check_year}.nc"
    if not proc_file.exists():
        print("  ⚠️  Processed file missing; skipping visual check.")
        return
 
    # ── 1. Find peak hourly value in the processed file for this month ─
    ds_proc     = xr.open_dataset(proc_file)
    month_start = f"{check_year}-{check_month:02d}-01"
    month_end   = (pd.Timestamp(month_start) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
    tp_month    = ds_proc["tp"].sel(time=slice(month_start, month_end))
    tp_month_mm = np.expm1(tp_month.values.astype(np.float64))
 
    # argmax across all dims → peak pixel + timestep
    flat_peak   = int(np.nanargmax(tp_month_mm))
    nt, nlat_p, nlon_p = tp_month_mm.shape
    ti_peak  = flat_peak // (nlat_p * nlon_p)
    rem      = flat_peak %  (nlat_p * nlon_p)
    lai_peak = rem // nlon_p
    loi_peak = rem %  nlon_p
 
    peak_time = pd.Timestamp(tp_month["time"].values[ti_peak])
    lat_peak  = float(tp_month["latitude"].values[lai_peak])
    lon_peak  = float(tp_month["longitude"].values[loi_peak])
    peak_val  = float(tp_month_mm[ti_peak, lai_peak, loi_peak])
 
    print(f"  Peak hourly rainfall : {peak_val:.2f} mm/hr")
    print(f"  At                   : {peak_time}  lat={lat_peak:.2f}  lon={lon_peak:.2f}")
 
    # ── 2. Define window snapped to day boundaries ─────────────────────
    # Snap back to the 00:00 of (peak_day - half_before)
    win_start = pd.Timestamp(peak_time.date()) - pd.Timedelta(days=half_before)
    win_end   = win_start + pd.Timedelta(days=n_days)   # exclusive
 
    _sep(
        f"VISUAL — {check_year}-{check_month:02d}  |  peak {peak_val:.1f} mm/hr @ "
        f"{peak_time.strftime('%m-%d %H:00')}  |  window {win_start.date()} → "
        f"{(win_end - pd.Timedelta(hours=1)).date()}  ({n_days} days)"
    )
 
    # ── 3. Extract processed series for this pixel over the window ─────
    tp_proc_log = ds_proc["tp"].sel(
        time=slice(str(win_start), str(win_end - pd.Timedelta(hours=1)))
    )
    tp_proc_mm  = np.expm1(
        tp_proc_log.sel(latitude=lat_peak, longitude=lon_peak, method="nearest")
        .values.astype(np.float64)
    )
    times_proc  = tp_proc_log["time"].values
    ds_proc.close()
 
    # ── 4. Load raw data, potentially spanning two monthly files ───────
    def _load_raw_month(year, month):
        f = RAW_OUTPUT / f"era5_land_tp_{year}_{month:02d}.nc"
        if not f.exists():
            return None, None
        ds = xr.open_dataset(f)
        ds = _clean_ds(ds)
        ds = _crop(ds)
        t  = ds["tp"]["time"].values
        v  = (
            ds["tp"]
            .sel(latitude=lat_peak, longitude=lon_peak, method="nearest")
            .values.astype(np.float64)
        ) * 1000.0   # m → mm
        ds.close()
        return t, v
 
    # Collect all (year, month) pairs covered by the raw window.
    # Raw window needs one extra hour before win_start for the boundary diff,
    # and runs to win_end (inclusive of the last 00:00 for mass-balance panel).
    raw_win_start = win_start - pd.Timedelta(hours=1)
    raw_win_end   = win_end
 
    months_needed = set()
    cur = pd.Timestamp(raw_win_start.year, raw_win_start.month, 1)
    while cur <= raw_win_end:
        months_needed.add((cur.year, cur.month))
        cur += pd.offsets.MonthBegin(1)
 
    raw_times_list, raw_vals_list = [], []
    for yr, mo in sorted(months_needed):
        t, v = _load_raw_month(yr, mo)
        if t is not None:
            raw_times_list.append(t)
            raw_vals_list.append(v)
 
    if not raw_times_list:
        print("  ⚠️  No raw files found for window; skipping visual check.")
        return
 
    raw_times_all = np.concatenate(raw_times_list)
    raw_vals_all  = np.concatenate(raw_vals_list)
 
    # Slice to [raw_win_start, raw_win_end]
    raw_mask   = (raw_times_all >= np.datetime64(raw_win_start)) & \
                 (raw_times_all <= np.datetime64(raw_win_end))
    times_raw  = raw_times_all[raw_mask]
    vals_raw_m = raw_vals_all[raw_mask]
 
    # Visible raw window for plotting: win_start → win_end-1h (exclude boundary hour)
    plot_mask  = raw_times_all >= np.datetime64(win_start)
    plot_mask &= raw_times_all <= np.datetime64(win_end - pd.Timedelta(hours=1))
    times_raw_plot = raw_times_all[plot_mask]
    vals_raw_plot  = raw_vals_all[plot_mask]
 
    # ── Plot ───────────────────────────────────────────────────────────
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(
        f"ERA5-Land TP verification — lat={lat_peak:.2f}°N, lon={lon_peak:.2f}°E\n"
        f"{check_year}-{check_month:02d}  |  peak {peak_val:.1f} mm/hr @ "
        f"{peak_time.strftime('%b %d %H:00 UTC')}  |  {n_days}-day window",
        fontsize=12,
    )
 
    def mark_resets(ax):
        for t in times_raw_plot:
            if pd.Timestamp(t).hour == 1:
                ax.axvline(pd.Timestamp(t), color="red", alpha=0.3,
                           linewidth=0.8, linestyle="--", label="_")
 
    def mark_peak(ax):
        ax.axvline(peak_time, color="purple", alpha=0.6,
                   linewidth=1.2, linestyle="-", label=f"Peak {peak_val:.1f} mm/hr")
 
    # Panel 1 — raw accumulated (mm)
    ax1.plot(pd.DatetimeIndex(times_raw_plot), vals_raw_plot,
             color="steelblue", linewidth=1.2, label="Raw accumulated TP (mm)")
    mark_resets(ax1)
    mark_peak(ax1)
    ax1.axvline(pd.Timestamp(times_raw_plot[0]), color="red", alpha=0.3,
                linewidth=0.8, linestyle="--", label="01:00 UTC reset")
    ax1.set_ylabel("Accumulated TP (mm)")
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)
 
    # Panel 2 — hourly processed (mm/hr)
    ax2.bar(pd.DatetimeIndex(times_proc), tp_proc_mm,
            width=pd.Timedelta(hours=1), color="darkorange",
            alpha=0.8, label="Processed hourly TP (mm/hr)")
    mark_resets(ax2)
    mark_peak(ax2)
    ax2.set_ylabel("Hourly TP (mm/hr)")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
 
    # Panel 3 — piecewise cumulative processed vs raw delta (both reset each day)
    # run_base must be the raw value AT 01:00 (start of the new run), not at
    # 00:00 (end of the previous run).  The old code used vals_raw_plot[i-1]
    # (the 00:00 value), which could be hundreds of mm after a large storm,
    # making every subsequent point in that run appear deeply negative.
    run_base = 0.0
    raw_zeroed = np.empty_like(vals_raw_plot)
    for i in range(len(times_raw_plot)):
        h = pd.Timestamp(times_raw_plot[i]).hour
        if h == 1:
            run_base = vals_raw_plot[i]   # value AT 01:00, i.e. the reset value itself
        raw_zeroed[i] = vals_raw_plot[i] - run_base
 
    proc_hours_arr    = np.array([pd.Timestamp(t).hour for t in times_proc])
    cumsum_piecewise  = np.zeros_like(tp_proc_mm)
    running = 0.0
    for i in range(len(tp_proc_mm)):
        if proc_hours_arr[i] == 1 and i > 0:
            running = 0.0
        running += tp_proc_mm[i]
        cumsum_piecewise[i] = running
 
    ax3.plot(pd.DatetimeIndex(times_proc), cumsum_piecewise,
             color="darkgreen", linewidth=1.4, label="Cumulative processed (mm, per run)")
    raw_plot_len = min(len(times_raw_plot), len(times_proc))
    ax3.plot(pd.DatetimeIndex(times_raw_plot[:raw_plot_len]), raw_zeroed[:raw_plot_len],
             color="steelblue", linewidth=1.0, linestyle=":",
             label="Raw accumulated (mm, zero-based per run)")
    mark_resets(ax3)
    mark_peak(ax3)
    ax3.set_ylabel("Piecewise cumulative TP (mm)")
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.xticks(rotation=30, ha="right")
 
    plt.tight_layout()
    out_png = PROCESSED_OUTPUT / (
        f"tp_verification_{check_year}_{check_month:02d}"
        f"_peak{peak_time.strftime('%m%d_%H')}h.png"
    )
    plt.savefig(str(out_png), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✅  Verification plot saved → {out_png}")


# ══════════════════════════════════════════════════════════════════════════
# Main runner
# ══════════════════════════════════════════════════════════════════════════

def run_all(
    visual_year: int  = 2020,
    visual_month: int = 8,
    visual_ndays: int = 5,
) -> None:
    """
    Run all checks in order.  Checks 1+2 share one raw-file pass.
    Downstream checks are still run even if earlier ones fail, so you
    get a complete picture in one run.
    """
    results = {}

    c1, c2 = check1_and_2_combined()
    results["check1_false_resets"] = c1
    results["check2_reset_hours"]  = c2

    results["check3_spikes"]       = check3_spike_scan()
    results["check4_mass_balance"] = check4_mass_balance()
    results["check5_timestamps"]   = check5_timestamp_alignment()
    results["check6_distribution"] = check6_distribution()

    visual_verification(visual_year, visual_month, visual_ndays)

    _sep("SUMMARY")
    all_ok = True
    for name, passed in results.items():
        icon = "✅" if passed else "❌"
        print(f"  {icon}  {name}")
        all_ok = all_ok and passed
    print()
    if all_ok:
        print("  🎉  All checks passed — preprocessing pipeline looks clean.")
    else:
        print("  ⚠️   One or more checks failed — review output above.")


if __name__ == "__main__":
    run_all(visual_year=2020, visual_month=8, visual_ndays=5)


### Visualize

In [ ]:
# ── Paste this into a new notebook cell and run independently ─────────────
# No need to run the full verification suite first.

import gc
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# ── Config — edit to match your setup ─────────────────────────────────────
RAW_OUTPUT       = Path("/mnt/data/khaiht/data/vietnam/output")
PROCESSED_OUTPUT = Path("/mnt/data/khaiht/data/vietnamvip_processed/output")

LAT_MIN, LAT_MAX = 5.8, 25.0
LON_MIN, LON_MAX = 102.0, 118.0

# ── Helpers ────────────────────────────────────────────────────────────────

def _clean_ds(ds):
    if "valid_time" in ds.dims or "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    for dim in ("expver", "number"):
        if dim in ds.dims:
            ds = ds.isel({dim: 0}, drop=True)
        if dim in ds.coords:
            ds = ds.drop_vars(dim)
    ds.attrs = {}
    for v in ds.variables:
        ds[v].attrs = {}
    return ds

def _crop(ds):
    return ds.sel(
        latitude=slice(LAT_MAX, LAT_MIN),
        longitude=slice(LON_MIN, LON_MAX),
    )

# ── Main function ──────────────────────────────────────────────────────────

def visual_verification(
    check_year: int  = 2020,
    check_month: int = 8,
    n_days: int      = 5,
) -> None:
    """
    3-panel verification plot centred on the peak hourly rainfall event
    within check_year / check_month:

      Panel 1 — raw accumulated TP (mm) with daily reset markers
      Panel 2 — processed hourly TP (mm/hr)
      Panel 3 — piecewise cumulative processed vs raw delta (mass-balance)

    The window is  [peak − floor(n_days/2) days,  peak + ceil(n_days/2) days),
    snapped to whole-day boundaries so reset markers always align cleanly.
    If the window extends into the adjacent month the raw data is loaded from
    both monthly files and concatenated.
    """
    half_before = n_days // 2

    proc_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{check_year}.nc"
    if not proc_file.exists():
        print(f"  ⚠️  Processed file not found: {proc_file}")
        return

    # ── 1. Find peak hourly value in the processed file for this month ─────
    ds_proc     = xr.open_dataset(proc_file)
    month_start = f"{check_year}-{check_month:02d}-01"
    month_end   = (pd.Timestamp(month_start) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
    tp_month    = ds_proc["tp"].sel(time=slice(month_start, month_end))
    tp_month_mm = np.expm1(tp_month.values.astype(np.float64))

    flat_peak      = int(np.nanargmax(tp_month_mm))
    nt, nlat_p, nlon_p = tp_month_mm.shape
    ti_peak  = flat_peak // (nlat_p * nlon_p)
    rem      = flat_peak %  (nlat_p * nlon_p)
    lai_peak = rem // nlon_p
    loi_peak = rem %  nlon_p

    peak_time = pd.Timestamp(tp_month["time"].values[ti_peak])
    lat_peak  = float(tp_month["latitude"].values[lai_peak])
    lon_peak  = float(tp_month["longitude"].values[loi_peak])
    peak_val  = float(tp_month_mm[ti_peak, lai_peak, loi_peak])

    print(f"  Peak hourly rainfall : {peak_val:.2f} mm/hr")
    print(f"  At                   : {peak_time}  lat={lat_peak:.2f}  lon={lon_peak:.2f}")

    # ── 2. Window snapped to day boundaries ───────────────────────────────
    win_start = pd.Timestamp(peak_time.date()) - pd.Timedelta(days=half_before)
    win_end   = win_start + pd.Timedelta(days=n_days)   # exclusive

    print(f"  Window               : {win_start.date()} → {(win_end - pd.Timedelta(hours=1)).date()}")

    # ── 3. Processed series for this pixel over the window ────────────────
    tp_proc_log = ds_proc["tp"].sel(
        time=slice(str(win_start), str(win_end - pd.Timedelta(hours=1)))
    )
    tp_proc_mm = np.expm1(
        tp_proc_log.sel(latitude=lat_peak, longitude=lon_peak, method="nearest")
        .values.astype(np.float64)
    )
    times_proc = tp_proc_log["time"].values
    ds_proc.close()

    # ── 4. Raw data — may span two monthly files ───────────────────────────
    def _load_raw_month(year, month):
        f = RAW_OUTPUT / f"era5_land_tp_{year}_{month:02d}.nc"
        if not f.exists():
            print(f"  ⚠️  Raw file not found: {f.name}")
            return None, None
        ds = xr.open_dataset(f)
        ds = _clean_ds(ds)
        ds = _crop(ds)
        t  = ds["tp"]["time"].values
        v  = (
            ds["tp"]
            .sel(latitude=lat_peak, longitude=lon_peak, method="nearest")
            .values.astype(np.float64)
        ) * 1000.0   # m → mm
        ds.close()
        return t, v

    raw_win_start = win_start - pd.Timedelta(hours=1)
    raw_win_end   = win_end

    months_needed = set()
    cur = pd.Timestamp(raw_win_start.year, raw_win_start.month, 1)
    while cur <= raw_win_end:
        months_needed.add((cur.year, cur.month))
        cur += pd.offsets.MonthBegin(1)

    raw_times_list, raw_vals_list = [], []
    for yr, mo in sorted(months_needed):
        t, v = _load_raw_month(yr, mo)
        if t is not None:
            raw_times_list.append(t)
            raw_vals_list.append(v)

    if not raw_times_list:
        print("  ⚠️  No raw files found for window; aborting.")
        return

    raw_times_all = np.concatenate(raw_times_list)
    raw_vals_all  = np.concatenate(raw_vals_list)

    plot_mask      = (raw_times_all >= np.datetime64(win_start)) & \
                     (raw_times_all <= np.datetime64(win_end - pd.Timedelta(hours=1)))
    times_raw_plot = raw_times_all[plot_mask]
    vals_raw_plot  = raw_vals_all[plot_mask]

    # ── 5. Panel 3 arrays ─────────────────────────────────────────────────
    # Raw zero-based: subtract the value at each 01:00 (run start) from that run
    run_base   = 0.0
    raw_zeroed = np.empty_like(vals_raw_plot)
    for i in range(len(times_raw_plot)):
        if pd.Timestamp(times_raw_plot[i]).hour == 1:
            run_base = vals_raw_plot[i]
        raw_zeroed[i] = vals_raw_plot[i] - run_base

    # Cumulative processed: reset to 0 at each 01:00
    proc_hours_arr   = np.array([pd.Timestamp(t).hour for t in times_proc])
    cumsum_piecewise = np.zeros_like(tp_proc_mm)
    running = 0.0
    for i in range(len(tp_proc_mm)):
        if proc_hours_arr[i] == 1 and i > 0:
            running = 0.0
        running += tp_proc_mm[i]
        cumsum_piecewise[i] = running

    # ── 6. Plot ───────────────────────────────────────────────────────────
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(
        f"ERA5-Land TP verification — lat={lat_peak:.2f}°N, lon={lon_peak:.2f}°E\n"
        f"{check_year}-{check_month:02d}  |  peak {peak_val:.1f} mm/hr @ "
        f"{peak_time.strftime('%b %d %H:00 UTC')}  |  {n_days}-day window",
        fontsize=12,
    )

    def mark_resets(ax):
        for t in times_raw_plot:
            if pd.Timestamp(t).hour == 1:
                ax.axvline(pd.Timestamp(t), color="red", alpha=0.3,
                           linewidth=0.8, linestyle="--", label="_")

    def mark_peak(ax):
        ax.axvline(peak_time, color="purple", alpha=0.6,
                   linewidth=1.2, linestyle="-", label=f"Peak {peak_val:.1f} mm/hr")

    # Panel 1 — raw accumulated
    ax1.plot(pd.DatetimeIndex(times_raw_plot), vals_raw_plot,
             color="steelblue", linewidth=1.2, label="Raw accumulated TP (mm)")
    mark_resets(ax1)
    mark_peak(ax1)
    ax1.axvline(pd.Timestamp(times_raw_plot[0]), color="red", alpha=0.3,
                linewidth=0.8, linestyle="--", label="01:00 UTC reset")
    ax1.set_ylabel("Accumulated TP (mm)")
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)

    # Panel 2 — processed hourly
    ax2.bar(pd.DatetimeIndex(times_proc), tp_proc_mm,
            width=pd.Timedelta(hours=1), color="darkorange",
            alpha=0.8, label="Processed hourly TP (mm/hr)")
    mark_resets(ax2)
    mark_peak(ax2)
    ax2.set_ylabel("Hourly TP (mm/hr)")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    # Panel 3 — mass-balance
    ax3.plot(pd.DatetimeIndex(times_proc), cumsum_piecewise,
             color="darkgreen", linewidth=1.4, label="Cumulative processed (mm, per run)")
    raw_plot_len = min(len(times_raw_plot), len(times_proc))
    ax3.plot(pd.DatetimeIndex(times_raw_plot[:raw_plot_len]), raw_zeroed[:raw_plot_len],
             color="steelblue", linewidth=1.0, linestyle=":",
             label="Raw accumulated (mm, zero-based per run)")
    mark_resets(ax3)
    mark_peak(ax3)
    ax3.set_ylabel("Piecewise cumulative TP (mm)")
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.xticks(rotation=30, ha="right")

    plt.tight_layout()
    out_png = PROCESSED_OUTPUT / (
        f"tp_verification_{check_year}_{check_month:02d}"
        f"_peak{peak_time.strftime('%m%d_%H')}h.png"
    )
    plt.savefig(str(out_png), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✅  Plot saved → {out_png}")


# ── Run ────────────────────────────────────────────────────────────────────
visual_verification(check_year=2024, check_month=5, n_days=5)

## Input dataset

In [ ]:
# ------------------------
# ERA5 INPUT PROCESSING (CLEAN VERSION)
# ------------------------

PL_VARS = {
    "omega500": 500,
    "z500": 500,
    "t850": 850,
    "q850": 850,
    "u850": 850,
    "v850": 850,
}

SL_VARS = ["t2m", "d2m", "msl", "tcwv", "u10", "v10"]

LAT_MIN, LAT_MAX = 0, 30
LON_MIN, LON_MAX = 95, 135


def load_and_process_var(var, year):
    print(f"  Loading {var} {year}")

    is_pl = var in PL_VARS
    fn = RAW_INPUT / f"era5_{'pl' if is_pl else 'sl'}_{var}_{year}.nc"
    print("   file:", fn)

    ds = xr.open_dataset(fn)
    ds = clean_ds(ds)

    # ------------------------
    # Pressure level handling (ROBUST)
    # ------------------------
    if is_pl:
        assert "pressure_level" in ds.dims, \
            f"{var} missing pressure_level dim"

        # ERA5 file already has only 1 level → just take index 0
        ds = ds.isel(pressure_level=0, drop=True)

    # ------------------------
    # Crop (full box)
    # ------------------------
    ds = crop(ds)
    print("   After crop dims:", ds.dims)

    # ------------------------
    # Rename data variable
    # ------------------------
    data_vars = list(ds.data_vars)
    assert len(data_vars) == 1, data_vars
    vname = data_vars[0]
    ds = ds.rename({vname: var})

    # ------------------------
    # Final sanity
    # ------------------------
    assert "time" in ds.dims
    assert "latitude" in ds.dims
    assert "longitude" in ds.dims
    assert "pressure_level" not in ds.dims

    return ds

for year in range(2017, 2026):
    print("=" * 80)
    print(f"Processing ERA5 INPUT year {year}")

    ds_list = []

    # pressure level vars
    for var in PL_VARS:
        ds_var = load_and_process_var(var, year)
        ds_list.append(ds_var)

    # single level vars
    for var in SL_VARS:
        ds_var = load_and_process_var(var, year)
        ds_list.append(ds_var)

    print("  Merging all variables...")
    ds_year = xr.merge(ds_list, compat="override", join="exact")

    print("  Final merged dataset:")
    print(ds_year)

    # ------------------------
    # Enforce float32
    # ------------------------
    ds_year = ds_year.astype("float32")

    # ------------------------
    # Final global sanity
    # ------------------------
    expected_dims = {"time", "latitude", "longitude"}
    assert set(ds_year.dims.keys()) == expected_dims, ds_year.dims

    out_fn = PROCESSED_INPUT / f"era5_input_{year}.nc"
    print("  Saving to:", out_fn)

    ds_year.to_netcdf(out_fn)

    print(f"  DONE year {year}")

## Inspect

In [ ]:
import xarray as xr

fn = "/mnt/data/khaiht/data/vietnam_processed/input/era5_input_2017.nc"
ds = xr.open_dataset(fn)

print(ds)

In [ ]:
print(list(ds.data_vars))

In [ ]:
print(ds.time.values[:3])
print(ds.time.values[-3:])
print("N timesteps:", ds.sizes["time"])

In [ ]:
print("Lat min/max:", float(ds.latitude.min()), float(ds.latitude.max()))
print("Lon min/max:", float(ds.longitude.min()), float(ds.longitude.max()))


In [ ]:
import xarray as xr
from pathlib import Path

ds = xr.open_dataset("data/processed/era5_land/era5_land_2019.nc")
print(ds)
print("NaN fraction:",
      ds["t2m"].isnull().mean().item())

# Combine into one nc file

## Target dataset

In [ ]:
from pathlib import Path
import xarray as xr

TRAIN_OUT = Path("/mnt/data/khaiht/data/vietnam_train/output")
TEST_OUT  = Path("/mnt/data/khaiht/data/vietnam_test/output")

TRAIN_OUT.mkdir(parents=True, exist_ok=True)
TEST_OUT.mkdir(parents=True, exist_ok=True)

print("Train folder:", TRAIN_OUT)
print("Test folder :", TEST_OUT)

In [ ]:
def merge_train_years(train_years):
    files = [
        PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{y}.nc"
        for y in train_years
        if (PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{y}.nc").exists()
    ]

    print(f"Merging train years: {train_years}")
    print(f"Found {len(files)} train files")

    ds_train = xr.open_mfdataset(files, combine="by_coords")
    ds_train = clean_ds(ds_train)

    out_file = TRAIN_OUT / f"era5_land_tp_vietnam_{train_years[0]}_{train_years[-1]}.nc"
    ds_train.to_netcdf(out_file)

    print(f"✅ Saved TRAIN file: {out_file}")
    return out_file


train_years = list(range(2017, 2025))  # 2018–2024
train_file = merge_train_years(train_years)

In [ ]:
def prepare_test_year(year):
    in_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    assert in_file.exists(), f"Missing test file: {in_file}"

    print(f"Preparing TEST year {year}")

    ds_test = xr.open_dataset(in_file)
    ds_test = clean_ds(ds_test)

    out_file = TEST_OUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_test.to_netcdf(out_file)

    print(f"✅ Saved TEST file: {out_file}")
    return out_file


test_file = prepare_test_year(2025)

In [ ]:
import xarray as xr

ds_tr = xr.open_dataset("/mnt/data/khaiht/data/vietnam_train/output/era5_land_tp_vietnam_2017_2024.nc")
ds_te = xr.open_dataset("/mnt/data/khaiht/data/vietnam_test/output/era5_land_tp_vietnam_2025.nc")

print("TRAIN time:", str(ds_tr.time.min().values), "→", str(ds_tr.time.max().values))
print("TEST  time:", str(ds_te.time.min().values), "→", str(ds_te.time.max().values))

print("TRAIN shape:", ds_tr.tp.shape)
print("TEST  shape:", ds_te.tp.shape)

## Input dataset

In [ ]:
from pathlib import Path
import xarray as xr

# ------------------------
# CONFIG
# ------------------------
PROCESSED_INPUT = Path("/mnt/data/khaiht/data/vietnam_processed/input")

TRAIN_IN = Path("/mnt/data/khaiht/data/vietnam_train/input")
TEST_IN  = Path("/mnt/data/khaiht/data/vietnam_test/input")

TRAIN_IN.mkdir(parents=True, exist_ok=True)
TEST_IN.mkdir(parents=True, exist_ok=True)

print("Train INPUT folder:", TRAIN_IN)
print("Test  INPUT folder:", TEST_IN)

In [ ]:
# ------------------------
# HELPERS
# ------------------------
def merge_train_input_years(train_years):
    out_file = TRAIN_IN / f"era5_input_{train_years[0]}_{train_years[-1]}.nc"

    print(f"Streaming merge INPUT train years: {train_years}")
    print("Output:", out_file)

    ds_out = None

    for y in train_years:
        fn = PROCESSED_INPUT / f"era5_input_{y}.nc"
        assert fn.exists(), f"❌ Missing INPUT year: {fn}"

        print("  Loading:", fn)
        ds_y = xr.open_dataset(fn, chunks={"time": 168})  # weekly chunks

        if ds_out is None:
            ds_out = ds_y
        else:
            ds_out = xr.concat([ds_out, ds_y], dim="time")

        print(f"   Appended {y}, total time now:", ds_out.sizes["time"])

    print("Writing TRAIN INPUT to disk (this can take several minutes)...")
    ds_out.to_netcdf(out_file)

    print(f"✅ Saved TRAIN INPUT file: {out_file}")
    return out_file

def prepare_test_input_year(year):
    in_file = PROCESSED_INPUT / f"era5_input_{year}.nc"
    assert in_file.exists(), f"❌ Missing test INPUT file: {in_file}"

    print(f"Preparing TEST INPUT year {year}")

    ds_test = xr.open_dataset(in_file)

    out_file = TEST_IN / f"era5_input_{year}.nc"
    ds_test.to_netcdf(out_file)

    print(f"✅ Saved TEST INPUT file: {out_file}")
    return out_file

In [ ]:
# ------------------------
# RUN
# ------------------------
train_years = list(range(2017, 2025))  # 2018–2024
train_in_file = merge_train_input_years(train_years)

In [ ]:
test_in_file = prepare_test_input_year(2025)

### Check

In [ ]:
train_in_file = Path("/mnt/data/khaiht/data/vietnam_train/input/era5_input_2017_2024.nc")
test_in_file = Path("/mnt/data/khaiht/data/vietnam_test/input/era5_input_2025.nc")

In [ ]:
# ------------------------
# SANITY CHECK
# ------------------------
ds_tr = xr.open_dataset(train_in_file)
ds_te = xr.open_dataset(test_in_file)

print("INPUT TRAIN time:",
      str(ds_tr.time.min().values), "→", str(ds_tr.time.max().values))
print("INPUT TEST  time:",
      str(ds_te.time.min().values), "→", str(ds_te.time.max().values))

print("INPUT TRAIN shape:", {k: ds_tr.sizes[k] for k in ds_tr.dims})
print("INPUT TEST  shape:",  {k: ds_te.sizes[k] for k in ds_te.dims})

print("INPUT variables:", list(ds_tr.data_vars))

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

ds = xr.open_dataset("data/final/era5_land_2019_2023.nc")

print(ds)
print("tp dtype:", ds["tp"].dtype)


In [ ]:
tp_max = ds["tp"].max(dim="time", skipna=True)

plt.figure()
tp_max.plot()
plt.title("ERA5 TP – max over time (2019–2023)")
plt.show()


In [ ]:
tp_p999 = ds["tp"].quantile(0.999, dim="time", skipna=True)

plt.figure()
tp_p999.plot()
plt.title("ERA5 TP – 99.9 percentile over time")
plt.show()


# Bilinear interpolation on ERA-Land fine grid

In [ ]:
!find /mnt/data/khaiht/data/vietnam_train_bilinear/vietnam_data_promax_m.zarr -type f -delete

In [ ]:
!find /mnt/data/khaiht/data/vietnam_train_bilinear/vietnam_data_promax_m.zarr -type d -empty -delete

In [ ]:
import xesmf
print(xesmf.__version__)

In [ ]:
import xarray as xr
import xesmf as xe

coarse = xr.open_dataset(
    "/mnt/data/khaiht/data/vietnam_train/input/era5_input_2017_2024.nc",
    chunks={"time": 24}
)

fine = xr.open_dataset(
    "/mnt/data/khaiht/data/vietnam_train/output/era5_land_tp_vietnam_2017_2024.nc",
    chunks={"time": 24}
)

# Take ONE timestep to define grid
coarse_grid = coarse.isel(time=0)
fine_grid   = fine.isel(time=0)

In [ ]:
# import xarray as xr
# import xesmf as xe

# coarse = xr.open_dataset(
#     "/mnt/data/khaiht/data/vietnam_test/input/era5_input_2025.nc",
#     chunks={"time": 24}
# )

# fine = xr.open_dataset(
#     "/mnt/data/khaiht/data/vietnam_test/output/era5_land_tp_vietnam_2025.nc",
#     chunks={"time": 24}
# )

# # Take ONE timestep to define grid
# coarse_grid = coarse.isel(time=0)
# fine_grid   = fine.isel(time=0)

In [ ]:
regridder = xe.Regridder(
    coarse_grid,
    fine_grid,
    method="bilinear",
    periodic=False,
    reuse_weights=False,
    filename="/mnt/data/khaiht/data/vietnam_train/regrid_weights.nc"
)

In [ ]:
regridder = xe.Regridder(
    coarse_grid,
    fine_grid,
    method="bilinear",
    periodic=False,
    reuse_weights=True,
    filename="/mnt/data/khaiht/data/vietnam_train/regrid_weights.nc"
)

coarse_on_fine = regridder(coarse)

In [ ]:
print(coarse_on_fine)

print("Lat range:",
      float(coarse_on_fine.latitude.min()),
      float(coarse_on_fine.latitude.max()))

print("Lon range:",
      float(coarse_on_fine.longitude.min()),
      float(coarse_on_fine.longitude.max()))

print("Vars:", list(coarse_on_fine.data_vars))

In [ ]:
print(fine)

print("Lat range:",
      float(fine.latitude.min()),
      float(fine.latitude.max()))

print("Lon range:",
      float(fine.longitude.min()),
      float(fine.longitude.max()))

print("Vars:", list(fine.data_vars))

In [ ]:
print(fine.latitude.equals(coarse_on_fine.latitude))
print(fine.longitude.equals(coarse_on_fine.longitude))
print(fine.time.equals(coarse_on_fine.time))

In [ ]:
print(set(coarse_on_fine.data_vars) & set(fine.data_vars))

In [ ]:
ds = xr.merge([coarse_on_fine, fine])

In [ ]:
# Create 2D land mask from first timestep ONLY
land_mask_2d = (~ds["tp"].isel(time=0).isnull()).astype("float32")

# Drop time dimension explicitly
land_mask_2d = land_mask_2d.drop_vars("time")

# Assign static land mask
# ds["land_mask"] = land_mask_2d
ds["tp"] = ds["tp"].fillna(0.0)

In [ ]:
from pathlib import Path

FINAL_TRAIN = Path("/mnt/data/khaiht/data/vietnam_train_bilinear")
FINAL_TRAIN.mkdir(parents=True, exist_ok=True)

## Compute normalization stats (only for training data)

In [ ]:
import xarray as xr
import json
import dask

out_stats = "/mnt/data/khaiht/data/vietnam_train_bilinear/stats.json"

input_vars = [
    "omega500", "z500", "t850", "q850", "u850", "v850",
    "t2m", "d2m", "msl", "tcwv", "u10", "v10"
]
output_vars = ["tp"]

stats = {"input": {}, "output": {}}

print("Computing INPUT stats...")
for v in input_vars:
    da = ds[v]
    mean, std = dask.compute(da.mean(), da.std())
    stats["input"][v] = {"mean": float(mean), "std": float(std)}

print("Computing OUTPUT stats...")
for v in output_vars:
    da = ds[v]
    mean, std = dask.compute(da.mean(), da.std())
    stats["output"][v] = {"mean": float(mean), "std": float(std)}

with open(out_stats, "w") as f:
    json.dump(stats, f, indent=2)

print("✅ Saved stats to:", out_stats)

## NetCDF (not recommended for large dataset)

In [ ]:
encoding = {}

for v in ds.data_vars:
    if "time" in ds[v].dims:
        encoding[v] = {
            "zlib": True,
            "complevel": 1,
            "dtype": "float32",
            "chunksizes": (24, ds.sizes["latitude"], ds.sizes["longitude"]),
        }
    else:
        # For 2D static vars like land_mask
        encoding[v] = {
            "zlib": True,
            "complevel": 1,
            "dtype": "float32",
            "chunksizes": (ds.sizes["latitude"], ds.sizes["longitude"]),
        }

In [ ]:
out_path = "/mnt/data/khaiht/data/vietnam_train_bilinear/vietnam_data.nc"

ds.to_netcdf(out_path, encoding=encoding)

print("✅ Saved:", out_path)

In [ ]:
import xarray as xr

# ds = xr.open_dataset(
#     "/mnt/data/khaiht/data/vietnam_train_bilinear/vietnam_data.nc"
# )

print(ds)
print("Vars:", list(ds.data_vars))
print("Lat:", ds.latitude.min().values, ds.latitude.max().values)
print("Lon:", ds.longitude.min().values, ds.longitude.max().values)
print("Time:", ds.time.min().values, ds.time.max().values)

In [ ]:
print("Vars:", list(ds.data_vars))
print("TP NaNs:", ds["tp"].isnull().sum())
print("Land mask mean:", float(ds["land_mask"].mean()))

## Zarr

In [ ]:
out_path = "/mnt/data/khaiht/data/vietnam_train_bilinear/vietnam_data_test.zarr"

# Rechunk for optimal CorrDiff access
ds_zarr = ds.chunk({
    "time": 1,   # one timestep per chunk (fast random access)
    "latitude": -1,
    "longitude": -1,
})

# Zarr encoding (no compression = fastest)
encoding = {
    v: {
        "compressor": None,
        "chunks": (1, ds.sizes["latitude"], ds.sizes["longitude"]),
        "dtype": "float32",
    }
    for v in ds_zarr.data_vars
}

ds_zarr.to_zarr(out_path, mode="w")

print("✅ Saved Zarr:", out_path)

In [ ]:
import xarray as xr

out_path = "/mnt/data/khaiht/data/vietnam_train_bilinear/vietnam_data_promax.zarr"
ds = xr.open_zarr(out_path, consolidated=True)
print(ds)

In [ ]:
print("Vars:", list(ds.data_vars))
print("TP NaNs:", ds["tp"].isnull().sum())

# Train / Validation / Test Split

## Recommended Split

| Split | Years | Samples (hrs) | Purpose |
|-------|-------|---------------|---------|
| **Train** | 2017–2023 | ~61,320 | Model fitting |
| **Validation** | 2024 | ~8,784 | Hyper-parameter tuning, early stopping |
| **Test** | 2025 | ~8,760 | Final unbiased evaluation |

### Why this is a good split
- **7 years of training** covers full seasonal and inter-annual variability (ENSO, monsoon cycles).
- **2024 as validation** is recent, so the model sees distribution shift before the final test.
- **2025 as held-out test** gives a true out-of-sample year; no data leakage.
- Chronological ordering is essential — random splits would cause *temporal leakage*.

### One minor consideration
ERA5-Land 0.1° has notable monthly maxima in 2020 (126 mm/hr), 2022 (114 mm/hr), 2021 (101 mm/hr). These extreme years are all in the training set, which is good for learning tail behaviour.


# Statistical Analysis of ERA5 & ERA5-Land Precipitation (2017–2025)

- **Fine (target):** ERA5-Land 0.1° — `era5_land_tp_vietnam_{year}.nc` (log1p mm/hr)
- **Coarse (input):** ERA5 single-level 0.25° — `era5_sl_tp_{year}.nc` (hourly mm/hr)


In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROCESSED_OUTPUT = Path("/mnt/data/khaiht/data/vietnamvip_processed/output")
RAW_INPUT_TP     = Path("/mnt/data/khaiht/data/vietnam/input_tp")

LAT_MIN, LAT_MAX = 5.8, 25.0
LON_MIN, LON_MAX = 102.0, 118.0

YEARS       = list(range(2017, 2026))
TRAIN_YEARS = list(range(2017, 2024))
VAL_YEARS   = [2024]
TEST_YEARS  = [2025]

print("Setup complete.")
print(f"Training  : {TRAIN_YEARS}")
print(f"Validation: {VAL_YEARS}")
print(f"Testing   : {TEST_YEARS}")


## Load processed datasets

In [ ]:
print("Loading ERA5-Land 0.1° processed files (2017–2025)...")
fine_files = [PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{y}.nc" for y in YEARS]
ds_fine    = xr.open_mfdataset(fine_files, combine="by_coords")
tp_fine_mm = np.expm1(ds_fine["tp"])
print(ds_fine)
print(f"\nFine grid shape: {dict(tp_fine_mm.sizes)}")
print(f"Time range: {str(tp_fine_mm.time.values[0])[:16]} → {str(tp_fine_mm.time.values[-1])[:16]}")
print(f"Min/Max (mm/hr): {float(tp_fine_mm.min()):.3f} / {float(tp_fine_mm.max()):.3f}")


In [ ]:
print("Loading ERA5 0.25° raw files (2017–2025)...")
coarse_list = []
for year in YEARS:
    fn  = RAW_INPUT_TP / f"era5_sl_tp_{year}.nc"
    ds_y = xr.open_dataset(fn)
    if "valid_time" in ds_y.dims:
        ds_y = ds_y.rename({"valid_time": "time"})
    tp_c = (ds_y["tp"] * 1000.0).clip(min=0.0)
    tp_c = tp_c.sel(latitude=slice(LAT_MAX, LAT_MIN),
                    longitude=slice(LON_MIN, LON_MAX))
    coarse_list.append(tp_c)
    print(f"  {year}: {dict(tp_c.sizes)}")

tp_coarse_mm = xr.concat(coarse_list, dim="time")

# ── Apply ERA5-Land land mask to coarse ERA5 ─────────────────────────────
# ERA5 covers ocean pixels that ERA5-Land marks NaN; without masking,
# coarse annual maxima can be driven by ocean convection, producing
# spurious fine/coarse ratios < 1 in some years.
_ds_mask = xr.open_dataset(PROCESSED_OUTPUT / "era5_land_tp_vietnam_2017.nc")
fine_land = (~_ds_mask["tp"].isel(time=0).isnull()).astype("float32")
_ds_mask.close()
fine_land_coarse = fine_land.interp(
    latitude=tp_coarse_mm.latitude,
    longitude=tp_coarse_mm.longitude,
    method="nearest",
)
tp_coarse_mm = tp_coarse_mm.where(fine_land_coarse > 0)
print("ERA5 coarse land-masked. Ocean NaN fraction:",
      f"{float(tp_coarse_mm.isel(time=0).isnull().mean()):.3f}")
print(f"\nCoarse grid shape: {dict(tp_coarse_mm.sizes)}")
print(f"Time range: {str(tp_coarse_mm.time.values[0])[:16]} → {str(tp_coarse_mm.time.values[-1])[:16]}")
print(f"Min/Max (mm/hr): {float(tp_coarse_mm.min()):.3f} / {float(tp_coarse_mm.max()):.3f}")


## 1. Annual Statistics Table

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. ANNUAL STATISTICS TABLE
# ══════════════════════════════════════════════════════════════
print("Computing annual statistics (fine 0.1° ERA5-Land)...")

records = []
for year in YEARS:
    fn   = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_y = xr.open_dataset(fn)
    tp   = np.expm1(ds_y["tp"])

    total_hrs = int(tp.sizes["time"])
    mean_val  = float(tp.mean())
    p99       = float(tp.quantile(0.99))
    p999      = float(tp.quantile(0.999))
    max_val   = float(tp.max())
    wet_frac  = float((tp > 0.1).mean())

    flat_idx = tp.argmax(dim=["time", "latitude", "longitude"])
    t_max    = str(tp.time.values[int(flat_idx["time"])])[:16]
    lat_m    = float(tp.latitude.values[int(flat_idx["latitude"])])
    lon_m    = float(tp.longitude.values[int(flat_idx["longitude"])])
    ds_y.close()

    split = "TRAIN" if year in TRAIN_YEARS else ("VAL" if year in VAL_YEARS else "TEST")
    records.append({
        "Year": year, "Split": split, "Hours": total_hrs,
        "Mean (mm/hr)": round(mean_val, 4),
        "p99 (mm/hr)": round(p99, 2),
        "p99.9 (mm/hr)": round(p999, 2),
        "Max (mm/hr)": round(max_val, 2),
        "Wet fraction": round(wet_frac, 4),
        "Peak time (UTC)": t_max,
        "Peak lat": round(lat_m, 2),
        "Peak lon": round(lon_m, 2),
    })

df_annual = pd.DataFrame(records)
print(df_annual.to_string(index=False))


## 2. Monthly Climatology — Fine vs Coarse

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. MONTHLY CLIMATOLOGY — fine vs coarse
# ══════════════════════════════════════════════════════════════
print("Computing monthly climatology...")

months      = list(range(1, 13))
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fine_monthly_mean, fine_monthly_max     = [], []
coarse_monthly_mean, coarse_monthly_max = [], []

tp_c_crop = tp_coarse_mm  # already cropped to Vietnam

for m in months:
    tp_m = tp_fine_mm.sel(time=tp_fine_mm.time.dt.month == m)
    fine_monthly_mean.append(float(tp_m.mean()))
    fine_monthly_max.append(float(tp_m.max()))

    tp_m_c = tp_c_crop.sel(time=tp_c_crop.time.dt.month == m)
    coarse_monthly_mean.append(float(tp_m_c.mean()))
    coarse_monthly_max.append(float(tp_m_c.max()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x, w = np.arange(12), 0.35
axes[0].bar(x-w/2, fine_monthly_mean,   w, label='ERA5-Land 0.1°', color='#2196F3', alpha=0.85)
axes[0].bar(x+w/2, coarse_monthly_mean, w, label='ERA5 0.25°',     color='#FF5722', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(month_names)
axes[0].set_ylabel('Mean rainfall (mm/hr)')
axes[0].set_title('Monthly mean rainfall over Vietnam, 2017–2025')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x-w/2, fine_monthly_max,   w, label='ERA5-Land 0.1°', color='#2196F3', alpha=0.85)
axes[1].bar(x+w/2, coarse_monthly_max, w, label='ERA5 0.25°',     color='#FF5722', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(month_names)
axes[1].set_ylabel('Max hourly rainfall (mm/hr)')
axes[1].set_title('Monthly maximum hourly rainfall — ERA5-Land captures higher extremes')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(pad=1.5)
plt.savefig(str(PROCESSED_OUTPUT / "stat_monthly_climatology.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Monthly climatology saved.")


## 3. Diurnal Cycle

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. DIURNAL CYCLE (Vietnam, all years)
# ══════════════════════════════════════════════════════════════
UTC_OFFSET = 7
print("Computing diurnal cycle...")

fine_diurnal, coarse_diurnal = [], []
for h in range(24):
    fine_diurnal.append(float(tp_fine_mm.sel(time=tp_fine_mm.time.dt.hour == h).mean()))
    coarse_diurnal.append(float(tp_c_crop.sel(time=tp_c_crop.time.dt.hour == h).mean()))

local_hours    = [(h + UTC_OFFSET) % 24 for h in range(24)]
order          = np.argsort(local_hours)
fine_d_local   = [fine_diurnal[i]   for i in order]
coarse_d_local = [coarse_diurnal[i] for i in order]
x_local        = sorted(local_hours)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_local, fine_d_local,   'o-',  color='#1565C0', lw=2, ms=5, label='ERA5-Land 0.1°')
ax.plot(x_local, coarse_d_local, 's--', color='#BF360C', lw=2, ms=5, label='ERA5 0.25°')
ax.set_xlabel('Local time (UTC+7)')
ax.set_ylabel('Mean hourly rainfall (mm/hr)')
ax.set_title('Diurnal cycle of precipitation over Vietnam, 2017–2025 (local time UTC+7)')
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], rotation=45)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(PROCESSED_OUTPUT / "stat_diurnal_cycle.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Diurnal cycle saved.")


## 4. Inter-annual Variability

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. INTER-ANNUAL VARIABILITY — annual domain maximum
# ══════════════════════════════════════════════════════════════
print("Computing inter-annual variability...")

ann_max_fine,  ann_max_coarse = [], []

for year in YEARS:
    fn   = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_y = xr.open_dataset(fn)
    tp_f = np.expm1(ds_y["tp"])
    ann_max_fine.append(float(tp_f.max()))

    tp_c_y = tp_coarse_mm.sel(time=tp_coarse_mm.time.dt.year == year)
    ann_max_coarse.append(float(tp_c_y.max()))
    ds_y.close()

ratios_max    = [f / max(c, 0.01) for f, c in zip(ann_max_fine, ann_max_coarse)]
ratio_max_str = f"{min(ratios_max):.1f}\u2013{max(ratios_max):.1f}\u00d7"

fig, ax = plt.subplots(figsize=(10, 5))

ax.axvspan(2017-0.5, 2023+0.5, alpha=0.07, color='blue',  label='Train')
ax.axvspan(2023+0.5, 2024+0.5, alpha=0.07, color='green', label='Val')
ax.axvspan(2024+0.5, 2025+0.5, alpha=0.07, color='red',   label='Test')
ax.set_xticks(YEARS); ax.tick_params(axis='x', rotation=45)
ax.grid(alpha=0.3)

ax.plot(YEARS, ann_max_fine,   'o-',  color='#1565C0', lw=2, ms=7, label='ERA5-Land 0.1°')
ax.plot(YEARS, ann_max_coarse, 's--', color='#BF360C', lw=2, ms=7, label='ERA5 0.25°')
ax.set_ylabel('Max hourly rainfall (mm/hr)')
ax.set_title(f"Annual domain maximum — ERA5-Land exceeds ERA5 by {ratio_max_str}")
ax.legend(fontsize=9)

plt.tight_layout(pad=1.5)
plt.savefig(str(PROCESSED_OUTPUT / "stat_interannual.png"), dpi=150, bbox_inches='tight')
plt.show()
print("\u2705 Inter-annual variability saved.")


## 5. Spatial Maps of Extreme Rainfall

In [ ]:
# ══════════════════════════════════════════════════════════════
# SHARED HELPER — Vietnam island markers (Hoàng Sa & Trường Sa)
# ══════════════════════════════════════════════════════════════

_VIETNAM_ISLANDS = [
    # (lat, lon, Vietnamese name, English name)
    (16.50, 112.00, "Hoàng Sa", "(Paracel Is.)"),
    ( 9.90, 114.20, "Trường Sa", "(Spratly Is.)"),
]

def _add_vietnam_islands(ax, transform, fontsize=6.5, marker_size=5):
    """
    Draw small yellow circle markers + bilingual labels for Hoàng Sa and
    Trường Sa on a cartopy axes.  Call once per map panel, after
    pcolormesh/imshow but before savefig.
    """
    for lat, lon, vname, ename in _VIETNAM_ISLANDS:
        ax.plot(
            lon, lat,
            marker="o", color="#FFEE44",
            markersize=marker_size,
            markeredgecolor="#333333", markeredgewidth=0.5,
            transform=transform, zorder=8,
        )
        ax.text(
            lon + 0.25, lat,
            f"{vname}\n{ename}",
            transform=transform,
            fontsize=fontsize,
            color="#111111",
            zorder=9,
            bbox=dict(
                facecolor="white", alpha=0.72,
                edgecolor="#999999", linewidth=0.4,
                pad=1.2, boxstyle="round,pad=0.3",
            ),
        )

print("Island-marker helper defined.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# SHARED HELPER — High-resolution (10 m) cartopy map features
# Uses NaturalEarthFeature at 10m so small islands (Hoàng Sa /
# Trường Sa archipelagos, Con Dao, Phú Quốc, …) are rendered.
# ══════════════════════════════════════════════════════════════
import cartopy.feature as cfeature

_LAND_10M = cfeature.NaturalEarthFeature(
    category="physical", name="land",
    scale="10m", facecolor="#f5f5f0", edgecolor="none", zorder=0
)
_COAST_10M = cfeature.NaturalEarthFeature(
    category="physical", name="coastline",
    scale="10m", facecolor="none", edgecolor="#222222", linewidth=0.8, zorder=1
)
_BORDERS_10M = cfeature.NaturalEarthFeature(
    category="cultural", name="admin_0_boundary_lines_land",
    scale="10m", facecolor="none", edgecolor="#555555", linewidth=0.6, zorder=2
)

print("High-resolution (10 m) cartopy features defined.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. SPATIAL MAPS — Multi-year max & p99 on Vietnam map
# ══════════════════════════════════════════════════════════════
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print("Computing spatial statistics maps...")
tp_fine_max = tp_fine_p99 = None
for year in YEARS:
    fn   = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_y = xr.open_dataset(fn)
    tp   = np.expm1(ds_y["tp"])
    if tp_fine_max is None:
        tp_fine_max = tp.max(dim="time").compute()
        tp_fine_p99 = tp.quantile(0.99, dim="time").compute()
    else:
        tp_fine_max = xr.concat([tp_fine_max, tp.max(dim="time").compute()],            dim="year").max(dim="year")
        tp_fine_p99 = xr.concat([tp_fine_p99, tp.quantile(0.99, dim="time").compute()], dim="year").max(dim="year")
    ds_y.close()

tp_coarse_max = tp_c_crop.max(dim="time").compute()
tp_coarse_p99 = tp_c_crop.quantile(0.99, dim="time").compute()

proj = ccrs.PlateCarree()
fig, axes = plt.subplots(2, 2, figsize=(14, 12), subplot_kw={'projection': proj})

datasets = [
    (tp_fine_max,   "ERA5-Land 0.1\u00b0 \u2014 Multi-year Max (mm/hr)",        "YlOrRd"),
    (tp_coarse_max, "ERA5 0.25\u00b0 \u2014 Multi-year Max (mm/hr)",             "YlOrRd"),
    (tp_fine_p99,   "ERA5-Land 0.1\u00b0 \u2014 99th Percentile (mm/hr)",        "Blues"),
    (tp_coarse_p99, "ERA5 0.25\u00b0 \u2014 99th Percentile (mm/hr)",            "Blues"),
]

for ax, (data, title, cmap) in zip(axes.flat, datasets):
    la = data.latitude.values; lo = data.longitude.values
    if la[0] > la[-1]:
        data = data.isel(latitude=slice(None, None, -1)); la = data.latitude.values
    im = ax.pcolormesh(lo, la, data.values, cmap=cmap, transform=proj, shading='auto')
    ax.add_feature(_BORDERS_10M)
    ax.add_feature(_COAST_10M)
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.5, linestyle='--')
    gl.top_labels = gl.right_labels = False
    plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.03, pad=0.04, label='mm/hr')
    ax.set_title(title, fontsize=10)
    _add_vietnam_islands(ax, transform=proj)   # Hoang Sa & Truong Sa

plt.suptitle("Spatial distribution of extreme rainfall over Vietnam, 2017\u20132025", fontsize=11, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.975])
plt.savefig(str(PROCESSED_OUTPUT / "stat_spatial_extremes.png"), dpi=150, bbox_inches='tight')
plt.show()
print("\u2705 Spatial maps saved.")


## 6. Rainfall Intensity Distribution (PDF & CCDF)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. RAINFALL INTENSITY DISTRIBUTION
#    NOTE: marginal PDF/CCDF comparisons between ERA5-Land (0.1°)
#    and ERA5 (0.25°) are not shown here.  Comparing marginal
#    distributions across two different spatial scales mixes the
#    effects of resolution with those of intensity bias: the
#    ERA5-Land distribution has ~1000× more pixels, each
#    representing a smaller area, so its tail is naturally heavier
#    purely from sampling a finer spatial field.  The correct
#    intensity comparison is the matched-pair sharpening ratio
#    shown via the extreme-events table (Section 7) and the
#    spatial maps (Section 5), where each fine value is compared
#    to the ERA5 value at the *same location and time*.
# ══════════════════════════════════════════════════════════════
print("Intensity distribution: see extreme-events table and spatial maps for matched-pair analysis.")

## 7. Top-10 Extreme Hourly Events

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. TOP-10 MOST EXTREME HOURLY EVENTS (domain-wide)
# ══════════════════════════════════════════════════════════════
# Both ERA5-Land 0.1° and ERA5 0.25° capture the same large-scale
# storm systems; the fine grid adds sub-grid spatial structure that
# concentrates intensity into smaller pixels, producing higher local
# peaks.  The ratio below reflects that spatial sharpening (typically
# 1.1–1.3×), not invisible sub-grid events.
print("Finding top-10 extreme hourly events...")
records_extreme = []
for year in YEARS:
    fn   = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_y = xr.open_dataset(fn)
    tp   = np.expm1(ds_y["tp"])
    max_val  = float(tp.max())
    flat_idx = tp.argmax(dim=["time", "latitude", "longitude"])
    t_max = str(tp.time.values[int(flat_idx["time"])])[:16]
    lat_m = float(tp.latitude.values[int(flat_idx["latitude"])])
    lon_m = float(tp.longitude.values[int(flat_idx["longitude"])])
    ds_y.close()

    t_dt    = pd.Timestamp(t_max)
    tp_snap = tp_coarse_mm.sel(time=t_dt, method="nearest")
    tp_c_pt     = float(tp_snap.sel(latitude=lat_m, longitude=lon_m,
                                    method="nearest").values)
    tp_c_dommax = float(tp_snap.max().values)

    records_extreme.append({
        "Year": year,
        "Time (UTC)": t_max,
        "Fine lat": round(lat_m, 1),
        "Fine lon": round(lon_m, 1),
        "Fine max (mm/hr)": round(max_val, 2),
        "Coarse @ pt (mm/hr)": round(tp_c_pt, 2),
        "Coarse dom-max (mm/hr)": round(tp_c_dommax, 2),
        "Ratio fine/coarse-max": round(max_val / (tp_c_dommax + 1e-6), 1),
    })

df_extreme = pd.DataFrame(records_extreme).sort_values("Fine max (mm/hr)", ascending=False)
print("\nTop extreme hourly rainfall events (ERA5-Land 0.1\u00b0 vs ERA5 0.25\u00b0, Vietnam, 2017\u20132025):")
print(df_extreme.to_string(index=False))

# Key insight derived from actual data
ratios      = df_extreme["Ratio fine/coarse-max"]
coarse_pts  = df_extreme["Coarse @ pt (mm/hr)"]
ratio_range = f"{ratios.min():.1f}\u2013{ratios.max():.1f}\u00d7"
pt_range    = f"{coarse_pts.min():.0f}\u2013{coarse_pts.max():.0f} mm/hr"

print("\nKey insight:")
print(f"  Both grids record substantial rainfall at the peak-pixel location: "
      f"Coarse @ pt = {pt_range}.")
print(f"  ERA5-Land 0.1\u00b0 exceeds ERA5 0.25\u00b0 domain-max by {ratio_range}, "
      f"reflecting finer spatial structure within storm cores,")
print( "  not invisible sub-grid events \u2014 both grids capture the same large-scale systems.")


## 8. Snapshot — Extreme Event: Fine vs Coarse

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. SNAPSHOT — Extreme event: Fine vs Coarse side-by-side
# ══════════════════════════════════════════════════════════════
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable

best        = df_extreme.iloc[0]
target_time = pd.Timestamp(best["Time (UTC)"])
target_lat  = best["Fine lat"]
target_lon  = best["Fine lon"]
target_year = int(best["Year"])

print(f"Visualising: {target_time}  |  Fine peak: {best['Fine max (mm/hr)']} mm/hr")

fn = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{target_year}.nc"
ds_y = xr.open_dataset(fn)
tp_fine_snap = np.expm1(ds_y["tp"].sel(time=target_time, method="nearest")).compute()
ds_y.close()

tp_coarse_snap = tp_coarse_mm.sel(time=target_time, method="nearest").sel(
    latitude=slice(LAT_MAX + 2, LAT_MIN - 2),
    longitude=slice(LON_MIN - 2, LON_MAX + 2)
).compute()

fine_max   = float(tp_fine_snap.max())
coarse_max = float(tp_coarse_snap.max())
ratio      = fine_max / max(coarse_max, 0.01)
coarse_at_peak = float(
    tp_coarse_snap.sel(latitude=target_lat, longitude=target_lon,
                       method="nearest").values
)

norm = mcolors.LogNorm(vmin=0.1, vmax=fine_max)
cmap = plt.cm.get_cmap("YlOrRd")
cmap.set_under("#f7f7f7")

proj = ccrs.PlateCarree()
fig, axes = plt.subplots(1, 2, figsize=(15, 7),
                          subplot_kw={'projection': proj},
                          gridspec_kw={'wspace': 0.18})

def make_map(ax, data, title, show_peak_dot=True):
    la = data.latitude.values; lo = data.longitude.values
    if la[0] > la[-1]:
        data = data.isel(latitude=slice(None, None, -1))
        la   = data.latitude.values
    im = ax.pcolormesh(lo, la, data.values, cmap=cmap, norm=norm,
                       transform=proj, shading='auto')
    ax.add_feature(_BORDERS_10M)
    ax.add_feature(_COAST_10M)
    ax.add_feature(_LAND_10M)
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.4,
                      linestyle='--', color='gray')
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = {'size': 8}
    gl.ylabel_style = {'size': 8}
    if show_peak_dot:
        ax.plot(target_lon, target_lat, marker='*', color='lime', ms=16,
                markeredgecolor='black', markeredgewidth=0.8,
                transform=proj, zorder=6)
    _add_vietnam_islands(ax, transform=proj)   # Hoang Sa & Truong Sa
    ax.set_title(title, fontsize=9, pad=7)
    return im

im1 = make_map(
    axes[0], tp_fine_snap,
    f"ERA5-Land 0.1° — {target_time.strftime('%Y-%m-%d %H:00 UTC')} | max: {fine_max:.1f} mm/hr (×{ratio:.2f} vs ERA5)",
)
im2 = make_map(
    axes[1], tp_coarse_snap,
    f"ERA5 0.25° — {target_time.strftime('%Y-%m-%d %H:00 UTC')} | max: {coarse_max:.1f} mm/hr (at ★: {coarse_at_peak:.1f} mm/hr)",
)

cbar = fig.colorbar(im1, ax=axes, orientation='vertical',
                    fraction=0.025, pad=0.03, shrink=0.85, extend='both')
cbar.set_label("Hourly rainfall intensity (mm/hr)", fontsize=10)
cbar.ax.tick_params(labelsize=8)

fig.suptitle(
    f"Extreme rainfall snapshot — {target_time.strftime('%Y-%m-%d %H:00 UTC')}  |  ERA5-Land: {fine_max:.1f} mm/hr vs ERA5: {coarse_max:.1f} mm/hr (×{ratio:.2f})  ★ = peak pixel",
    fontsize=11, y=0.98,
)

plt.savefig(str(PROCESSED_OUTPUT / "stat_extreme_snapshot.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("\u2705 Extreme snapshot saved.")


## 9. Animation — 24-hour Evolution of Extreme Event

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. ANIMATION — 24-hour evolution of an extreme event (GIF)
#    Layout: side-by-side (coarse left, fine right)
# ══════════════════════════════════════════════════════════════
import matplotlib.animation as animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

anim_year = int(best["Year"])
t_peak    = pd.Timestamp(best["Time (UTC)"])
t_start   = t_peak - pd.Timedelta(hours=12)
t_end     = t_peak + pd.Timedelta(hours=11)
print(f"Building animation {t_start} \u2192 {t_end}  ({anim_year})")

fn = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{anim_year}.nc"
ds_y = xr.open_dataset(fn)
tp_fine_anim = np.expm1(
    ds_y["tp"].sel(time=slice(str(t_start), str(t_end)))
).compute()
ds_y.close()

tp_coarse_anim = tp_coarse_mm.sel(
    time=slice(str(t_start), str(t_end))
).sel(
    latitude=slice(LAT_MAX + 2, LAT_MIN - 2),
    longitude=slice(LON_MIN - 2, LON_MAX + 2)
).compute()

n_frames      = tp_fine_anim.sizes["time"]
fine_anim_max = float(tp_fine_anim.max())
norm_anim     = mcolors.LogNorm(vmin=0.1, vmax=fine_anim_max)
cmap_anim     = plt.cm.get_cmap("YlOrRd")
cmap_anim.set_under("#f7f7f7")

proj = ccrs.PlateCarree()
fig  = plt.figure(figsize=(14, 7))
gs   = gridspec.GridSpec(1, 3, figure=fig,
                          width_ratios=[1, 1, 0.04], wspace=0.08)
ax_c  = fig.add_subplot(gs[0, 0], projection=proj)
ax_f  = fig.add_subplot(gs[0, 1], projection=proj)
ax_cb = fig.add_subplot(gs[0, 2])

for ax, label in [
    (ax_c, "ERA5 0.25\u00b0 (coarse input)"),
    (ax_f, "ERA5-Land 0.1\u00b0 (fine target)"),
]:
    ax.add_feature(_BORDERS_10M)
    ax.add_feature(_COAST_10M)
    ax.add_feature(_LAND_10M)
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4,
                      linestyle='--', color='gray')
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = {'size': 7}
    gl.ylabel_style = {'size': 7}
    ax.set_title(label, fontsize=10, pad=4)
    _add_vietnam_islands(ax, transform=proj, fontsize=6.0, marker_size=4)

def lats_lons_vals(da):
    la = da.latitude.values; lo = da.longitude.values
    if la[0] > la[-1]:
        da = da.isel(latitude=slice(None, None, -1)); la = da.latitude.values
    return lo, la, da.values

lo_c, la_c, v_c = lats_lons_vals(tp_coarse_anim.isel(time=0))
lo_f, la_f, v_f = lats_lons_vals(tp_fine_anim.isel(time=0))

mesh_c = ax_c.pcolormesh(lo_c, la_c, v_c, cmap=cmap_anim, norm=norm_anim,
                          transform=proj, shading='auto')
mesh_f = ax_f.pcolormesh(lo_f, la_f, v_f, cmap=cmap_anim, norm=norm_anim,
                          transform=proj, shading='auto')

peak_star, = ax_f.plot(
    best["Fine lon"], best["Fine lat"],
    marker='*', color='lime', ms=14,
    markeredgecolor='black', markeredgewidth=0.7,
    transform=proj, zorder=10,
)

import matplotlib as mpl
sm = mpl.cm.ScalarMappable(cmap=cmap_anim, norm=norm_anim)
sm.set_array([])
cbar = fig.colorbar(sm, cax=ax_cb, extend='both')
cbar.set_label('Rainfall (mm/hr)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

time_text  = fig.text(0.5, 0.98, '', ha='center', va='top',
                       fontsize=12, fontweight='bold',
                       transform=fig.transFigure)
coarse_ann = ax_c.text(0.02, 0.02, '', transform=ax_c.transAxes,
                        fontsize=8, color='#222',
                        bbox=dict(facecolor='white', alpha=0.7, pad=2, edgecolor='none'))
fine_ann   = ax_f.text(0.02, 0.02, '', transform=ax_f.transAxes,
                        fontsize=8, color='#222',
                        bbox=dict(facecolor='white', alpha=0.7, pad=2, edgecolor='none'))

def update(frame):
    snap_c = tp_coarse_anim.isel(time=frame)
    snap_f = tp_fine_anim.isel(time=frame)
    _, _, vc = lats_lons_vals(snap_c)
    _, _, vf = lats_lons_vals(snap_f)
    mesh_c.set_array(vc.ravel())
    mesh_f.set_array(vf.ravel())
    t_str = str(tp_fine_anim.time.values[frame])[:16].replace("T", " ") + " UTC"
    time_text.set_text(t_str)
    coarse_ann.set_text(f"Max: {float(snap_c.max()):.1f} mm/hr")
    fine_ann.set_text(  f"Max: {float(snap_f.max()):.1f} mm/hr")
    return mesh_c, mesh_f, time_text, coarse_ann, fine_ann

ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=400, blit=False)
gif_path = str(PROCESSED_OUTPUT / f"extreme_event_animation_{anim_year}.gif")
ani.save(gif_path, writer='pillow', fps=2.5, dpi=110)
print(f"\u2705 Animation saved: {gif_path}")
plt.close()


## 10. Top-4 Events Panel (for paper/poster)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. TOP-4 EVENTS PANEL — 2×4 figure for paper/poster
# ══════════════════════════════════════════════════════════════
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

top4 = df_extreme.head(4)
proj = ccrs.PlateCarree()
fig  = plt.figure(figsize=(18, 11))
gs   = gridspec.GridSpec(2, 4, figure=fig, hspace=0.08, wspace=0.05,
                         top=0.93)           # ← reserve exactly this much room for the title

norm_panel = mcolors.LogNorm(vmin=0.1, vmax=150)
cmap_panel = plt.cm.YlOrRd
cmap_panel.set_under("whitesmoke")

axes_top    = [fig.add_subplot(gs[0, c], projection=proj) for c in range(4)]
axes_bottom = [fig.add_subplot(gs[1, c], projection=proj) for c in range(4)]

for idx, row in enumerate(top4.itertuples()):
    year = int(row.Year)
    t_dt = pd.Timestamp(row._2)

    fn   = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_y = xr.open_dataset(fn)
    fine_snap = np.expm1(ds_y["tp"].sel(time=t_dt, method="nearest")).compute()
    ds_y.close()

    coarse_snap = tp_coarse_mm.sel(time=t_dt, method="nearest").sel(
        latitude=slice(LAT_MAX + 2, LAT_MIN - 2),
        longitude=slice(LON_MIN - 2, LON_MAX + 2)
    ).compute()

    def plot_panel(ax, data, title):
        la = data.latitude.values; lo = data.longitude.values
        if la[0] > la[-1]:
            data = data.isel(latitude=slice(None, None, -1)); la = data.latitude.values
        im = ax.pcolormesh(lo, la, data.values, cmap=cmap_panel, norm=norm_panel,
                           transform=proj, shading='auto')
        ax.add_feature(_BORDERS_10M)
        ax.add_feature(_COAST_10M)
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)
        gl = ax.gridlines(linewidth=0.3, alpha=0.4, linestyle='--')
        gl.top_labels = gl.right_labels = gl.bottom_labels = gl.left_labels = False
        ax.set_title(title, fontsize=8)
        _add_vietnam_islands(ax, transform=proj, fontsize=5.5, marker_size=4)
        return im

    im = plot_panel(
        axes_top[idx], coarse_snap,
        f"ERA5 0.25°  {t_dt.strftime('%Y-%m-%d %H:00')} UTC | {float(coarse_snap.max()):.1f} mm/hr",
    )
    im = plot_panel(
        axes_bottom[idx], fine_snap,
        f"ERA5-Land 0.1°  {t_dt.strftime('%Y-%m-%d %H:00')} UTC | {float(fine_snap.max()):.1f} mm/hr",
    )
    for ax in [axes_top[idx], axes_bottom[idx]]:
        ax.plot(row._5, row._4, 'w*', ms=10, transform=proj, zorder=10)

plt.colorbar(im, ax=axes_bottom, orientation='horizontal', fraction=0.015, pad=0.05,
             label='Hourly rainfall (mm/hr)', extend='both')

mean_ratio = df_extreme["Ratio fine/coarse-max"].mean()
plt.suptitle(
    f"Top-4 extreme hourly rainfall events over Vietnam, 2017–2025  |  mean fine/coarse ratio: ×{mean_ratio:.2f}  (★ = peak pixel)",
    fontsize=11, y=0.98,           # ← sits just above the gs top=0.93 boundary
)
plt.savefig(str(PROCESSED_OUTPUT / "stat_top4_events_panel.png"), dpi=150,
            bbox_inches='tight', pad_inches=0.05)   # ← small fixed pad instead of auto-expanding
plt.show()
print("✅ Top-4 panel saved.")

## 11. Split Summary & Recommendation

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. SPLIT SUMMARY TABLE
# ══════════════════════════════════════════════════════════════
print("=" * 70)
print("DATASET SPLIT SUMMARY")
print("=" * 70)

splits = {
    "TRAIN (2017\u20132023)": TRAIN_YEARS,
    "VAL   (2024)"           : VAL_YEARS,
    "TEST  (2025)"           : TEST_YEARS,
}

split_stats = {}
for split_name, years in splits.items():
    rows = df_annual[df_annual["Year"].isin(years)]
    print(f"\n{split_name}")
    print(f"  Years           : {years}")
    print(f"  Total hours     : {rows['Hours'].sum():,}")
    print(f"  Mean (mm/hr)    : {rows['Mean (mm/hr)'].mean():.4f}")
    print(f"  Max ever (mm/hr): {rows['Max (mm/hr)'].max():.2f}")
    print(f"  p99 range       : "
          f"{rows['p99 (mm/hr)'].min():.2f} \u2013 "
          f"{rows['p99 (mm/hr)'].max():.2f}")
    print(f"  Wet fraction    : {rows['Wet fraction'].mean():.4f}")
    split_stats[split_name] = rows

# Recommendation text derived from actual df_annual values
train_rows   = split_stats["TRAIN (2017\u20132023)"]
val_rows     = split_stats["VAL   (2024)"]
train_max    = train_rows["Max (mm/hr)"].max()
train_max_yr = int(train_rows.loc[train_rows["Max (mm/hr)"].idxmax(), "Year"])
val_max      = float(val_rows["Max (mm/hr)"].max())

print("\n\nRECOMMENDATION:")
print("  \u2705  2017\u20132023 train | 2024 val | 2025 test is a solid chronological split.")
print(f"  \u2705  Training-set peak is {train_max:.1f} mm/hr ({train_max_yr}) \u2014 "
      f"sufficient tail coverage for learning heavy-tail precipitation.")
print(f"  \u2705  2024 validation captures a recent year with {val_max:.1f} mm/hr max.")
print("  \u2705  2025 test is fully held-out; no temporal leakage possible.")
print("  \u26a0\ufe0f   The split is NOT random (required for time series) \u2014 do not shuffle.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# BASELINE METRICS — ERA5 bilinear interpolation vs ERA5-Land
# Computes MAE, CRPS, FSS, BS for train (2017–2023) and test (2025)
# using the same compute_metrics / aggregate_metrics interface
# as the model inference notebooks.
# ══════════════════════════════════════════════════════════════

# ── Cell 1 — Imports and config ───────────────────────────────────────────────

import warnings
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path
from scipy.interpolate import RegularGridInterpolator

from inference_shared import compute_metrics, aggregate_metrics, print_metrics_table

PROCESSED_OUTPUT = Path("/mnt/data/khaiht/data/vietnamvip_processed/output")
RAW_INPUT_TP     = Path("/mnt/data/khaiht/data/vietnam/input_tp")

LAT_MIN, LAT_MAX = 5.9, 25.0
LON_MIN, LON_MAX = 102.1, 118.0

THRESHOLD_MM = 10.0   # exceedance threshold for FSS and BS
FSS_SCALE    = 3      # neighbourhood half-width in grid cells

SPLITS = {
    "TRAIN (2017-2023)": list(range(2017, 2024)),
    "TEST  (2025)":      [2025],
}

print("Config ready.")


# ── Cell 2 — Build fine-grid coordinates and land mask ───────────────────────

# Load one processed file to get the fine grid and derive the land mask
_ds_ref = xr.open_dataset(PROCESSED_OUTPUT / "era5_land_tp_vietnam_2017.nc")
fine_lat = _ds_ref["latitude"].values   # (192,) descending
fine_lon = _ds_ref["longitude"].values  # (160,)
land_mask = (~_ds_ref["tp"].isel(time=0).isnull()).values.astype(bool)  # (192, 160)
_ds_ref.close()

# 2-D coordinate arrays for plotting (not needed for metrics, but handy)
LON2D, LAT2D = np.meshgrid(fine_lon, fine_lat)   # (192, 160)

print(f"Fine grid : {len(fine_lat)} × {len(fine_lon)}")
print(f"Land pixels: {land_mask.sum():,} / {land_mask.size:,}")


# ── Cell 3 — Helper: bilinearly interpolate one ERA5 snapshot to fine grid ───

def interpolate_era5_to_fine(tp_coarse_snap):
    """
    Bilinearly interpolate a single ERA5 coarse snapshot (DataArray with
    latitude/longitude coords) onto the fine ERA5-Land grid.

    Parameters
    ----------
    tp_coarse_snap : xr.DataArray  shape (n_lat_coarse, n_lon_coarse), mm/hr

    Returns
    -------
    np.ndarray  shape (192, 160), mm/hr, NaN where land_mask is False
    """
    c_lat = tp_coarse_snap.latitude.values
    c_lon = tp_coarse_snap.longitude.values

    # RegularGridInterpolator expects ascending axes
    if c_lat[0] > c_lat[-1]:
        vals = tp_coarse_snap.values[::-1, :]
        c_lat = c_lat[::-1]
    else:
        vals = tp_coarse_snap.values

    vals = np.nan_to_num(vals, nan=0.0)   # ocean NaNs → 0 before interpolation

    interp = RegularGridInterpolator(
        (c_lat, c_lon), vals,
        method="linear",
        bounds_error=False,
        fill_value=0.0,
    )

    # Fine-grid query points
    fine_lat_asc = fine_lat[::-1] if fine_lat[0] > fine_lat[-1] else fine_lat
    pts = np.array(np.meshgrid(fine_lat_asc, fine_lon, indexing="ij")).reshape(2, -1).T
    out = interp(pts).reshape(len(fine_lat_asc), len(fine_lon))

    # Restore descending latitude order if needed
    if fine_lat[0] > fine_lat[-1]:
        out = out[::-1, :]

    out = np.clip(out, 0.0, None)
    out = np.where(land_mask, out, np.nan)
    return out.astype(np.float32)

print("Interpolation helper defined.")


# ── Cell 4 — Load coarse ERA5 for both splits (land-masked) ──────────────────

def load_coarse_year(year):
    """Return land-masked ERA5 hourly TP (mm/hr) for one year as xr.DataArray."""
    fn  = RAW_INPUT_TP / f"era5_sl_tp_{year}.nc"
    ds  = xr.open_dataset(fn)
    if "valid_time" in ds.dims:
        ds = ds.rename({"valid_time": "time"})
    tp  = (ds["tp"] * 1000.0).clip(min=0.0)
    tp  = tp.sel(latitude=slice(LAT_MAX + 2, LAT_MIN - 2),
                 longitude=slice(LON_MIN - 2, LON_MAX + 2))
    ds.close()
    return tp

print("Coarse loader defined.")


# ── Cell 5 — Run metrics over each split ─────────────────────────────────────

results = {}

for split_name, years in SPLITS.items():
    print(f"\n{'='*60}")
    print(f"  {split_name}")
    print(f"{'='*60}")

    metrics_list = []

    for year in years:
        print(f"  Processing {year}...", end=" ", flush=True)

        # Load fine (target) for this year
        ds_fine = xr.open_dataset(PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc")
        tp_fine_log = ds_fine["tp"]          # log1p-transformed

        # Load coarse ERA5 for this year
        tp_coarse = load_coarse_year(year)

        times = tp_fine_log.time.values

        for t in times:
            # Fine target in physical units (mm/hr)
            truth_log = tp_fine_log.sel(time=t, method="nearest").values
            truth_mm  = np.expm1(truth_log).astype(np.float32)

            # Coarse ERA5 at same timestamp, interpolated to fine grid
            snap_coarse = tp_coarse.sel(time=t, method="nearest")
            pred_mm     = interpolate_era5_to_fine(snap_coarse)  # (192, 160)

            m = compute_metrics(
                truth_tp    = truth_mm,
                pred_all    = pred_mm,          # (H, W) → treated as single member
                land_mask   = land_mask,
                threshold_mm = THRESHOLD_MM,
                fss_scale   = FSS_SCALE,
            )
            metrics_list.append(m)

        ds_fine.close()
        print(f"done ({len(times)} timesteps)")

    agg = aggregate_metrics(metrics_list)
    results[split_name] = agg
    print_metrics_table(f"ERA5 interpolated  |  {split_name}", agg)

print("\n✅ Baseline metrics complete.")


# ── Cell 6 — Summary table ────────────────────────────────────────────────────

print("\n" + "═"*62)
print("  SUMMARY — ERA5 bilinear interpolation baseline")
print(f"  Threshold: {THRESHOLD_MM} mm/hr  |  FSS scale: {FSS_SCALE} grid cells")
print("═"*62)
print(f"  {'Split':<22}  {'MAE':>8}  {'CRPS':>8}  {'FSS':>8}  {'BS':>8}")
print(f"  {'-'*22}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}")
for split_name, m in results.items():
    print(f"  {split_name:<22}  {m['mae']:8.4f}  {m['crps']:8.4f}  {m['fss']:8.4f}  {m['bs']:8.4f}")
print("═"*62)

# Optionally save to CSV for later comparison with model results
df_results = pd.DataFrame(results).T
df_results.index.name = "split"
df_results.to_csv(PROCESSED_OUTPUT / "metrics_era5_interpolated_baseline.csv")
print("\nSaved to metrics_era5_interpolated_baseline.csv")